# Notebook 23 — `race_type_raw` semantics

## Purpose

This notebook investigates what `race_type_raw` actually represents in the accepted Inside Rails database and whether it is sufficiently understood for governed analytical use.

The investigation was triggered while working on a reader-facing study, but this is a **database/source-field semantics investigation**, so it belongs in the database notebook series under `notebooks/`.

## Bounded question

> What does `race_type_raw` mean, how consistently is it populated, and what — if anything — can safely be inferred from it?

We will begin from the literal stored values rather than from an assumed racing definition.

## Analytical source

Current accepted database:

`data/processed/database/releases/inside_rails_v3.sqlite3`

Database v3 is **accepted and immutable**. It must be opened read-only and must not be modified or rebuilt during this investigation.

Preferred race-level interface:

`view_reconciled_race_occurrences`

Expected grain:

> one reconciled Source Version 1 race occurrence

Expected population:

**189,043 races**

Read-only access uses:

```python
from inside_rails.source_sqlite import connect_read_only
```

## Working method

The investigation will proceed one question at a time:

1. establish coverage and literal values;
2. inspect what those values appear to encode;
3. test whether meaning varies by jurisdiction, period or other already-governed race context where necessary;
4. compare the field with existing race-classification governance;
5. inspect ambiguous or exceptional cases before interpreting them;
6. decide whether the existing field is analytically sufficient or whether additional governance is justified.

Raw values and lineage will be preserved throughout.

No canonical categories or racing taxonomy will be imposed before the evidence supports them.

If the investigation establishes a reusable or correctness-critical database consequence, that will be documented separately and handled through the normal database candidate, validation and release process. The accepted Database v3 release itself remains unchanged.


## 1. What is actually stored in `race_type_raw`?

Before interpreting `race_type_raw`, we need to establish its basic shape at race level.

This first check asks only:

* whether the field exists in the accepted race-level view;
* how many races contain a null or blank value;
* how many distinct literal values occur;
* which literal values are most common.

This is a profiling step, not an interpretation step.

A value such as `Handicap`, `Maiden` or any abbreviation will not yet be assumed to represent a single well-defined racing category merely because its label appears familiar.

The result will determine what question we ask next.


In [1]:
from pathlib import Path

import pandas as pd

from inside_rails.source_sqlite import connect_read_only


# Use the documented repository location rather than relying on the notebook's
# working directory. Database v3 is the accepted immutable analytical release.
PROJECT_ROOT = Path.home() / "Documents" / "inside-rails-horse-racing"
DATABASE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "database"
    / "releases"
    / "inside_rails_v3.sqlite3"
)

# Use the reconciled race-occurrence view because this investigation is at
# race grain and should inherit all governance already accepted into v3.
RACE_VIEW = "view_reconciled_race_occurrences"

# Fail immediately rather than silently falling back to another database.
assert DATABASE.is_file(), f"Database not found: {DATABASE}"

with connect_read_only(DATABASE) as connection:
    # Protect the accepted race-level population before examining race_type_raw.
    # A different count would mean we are not analysing the expected v3 race view.
    race_rows = connection.execute(
        f"SELECT COUNT(*) FROM {RACE_VIEW}"
    ).fetchone()[0]

    assert race_rows == 189_043, (
        f"Unexpected race population: {race_rows:,}"
    )

    # Confirm that race_type_raw is genuinely exposed at this governed
    # race-occurrence grain rather than assuming its presence from its name.
    columns = [
        row[1]
        for row in connection.execute(
            f"PRAGMA table_info({RACE_VIEW})"
        ).fetchall()
    ]

    assert "race_type_raw" in columns, (
        f"race_type_raw not found in {RACE_VIEW}"
    )

    # Profile missingness without cleaning or interpreting the field.
    # Nulls and zero-length/whitespace-only strings are counted separately
    # because they are different source states and may have different causes.
    coverage = pd.read_sql_query(
        f"""
        SELECT
            COUNT(*) AS races,
            SUM(
                CASE
                    WHEN race_type_raw IS NULL THEN 1
                    ELSE 0
                END
            ) AS null_races,
            SUM(
                CASE
                    WHEN race_type_raw IS NOT NULL
                     AND TRIM(CAST(race_type_raw AS TEXT)) = ''
                    THEN 1
                    ELSE 0
                END
            ) AS blank_races,
            COUNT(DISTINCT race_type_raw) AS distinct_raw_values
        FROM {RACE_VIEW}
        """,
        connection,
    )

    # Preserve the literal stored vocabulary. Do not trim, normalise, merge or
    # recode values yet: differences in spelling, case or formatting may be
    # evidence about the field's semantics or source behaviour.
    values = pd.read_sql_query(
        f"""
        SELECT
            race_type_raw,
            COUNT(*) AS races
        FROM {RACE_VIEW}
        GROUP BY race_type_raw
        ORDER BY races DESC, race_type_raw
        """,
        connection,
    )

# Keep both the population-level coverage and literal-value frequencies visible
# so the next analytical question is chosen from observed evidence.
display(coverage)
display(values.head(50))

,races,null_races,blank_races,distinct_raw_values
0,189043,0,0,4


,race_type_raw,races
0,Flat,126391
1,Hurdle,35462
2,Chase,22547
3,NH Flat,4643


### What we found

`race_type_raw` is completely populated across the accepted race-level population of **189,043 races**.

There are:

* **0 null values**
* **0 blank values**
* exactly **4 distinct literal values**

The four stored values are:

| `race_type_raw` |   Races | Share |
| --------------- | ------: | ----: |
| `Flat`          | 126,391 | 66.9% |
| `Hurdle`        |  35,462 | 18.8% |
| `Chase`         |  22,547 | 11.9% |
| `NH Flat`       |   4,643 |  2.5% |

### Interpretation

At source level, `race_type_raw` is therefore not a free-text field or a detailed race-description field. It behaves like a small, complete categorical field with four values.

The labels suggest that the field may be describing broad racing code or discipline rather than concepts such as handicap, maiden, novice, selling or claiming status.

That interpretation is **not yet authorised from the labels alone**.

The database already contains race-classification governance from the earlier notebook series, so before defining `race_type_raw` ourselves we need to establish how these four raw values relate to the governed classification fields already present in Database v3.

### What this does not establish

This result does not yet prove:

* the precise racing definition of each value;
* whether `Flat` includes both turf and all-weather racing;
* whether `NH Flat` should be treated as flat racing, National Hunt racing, or a separate analytical category;
* whether `Hurdle` and `Chase` completely describe jumps racing across every jurisdiction;
* whether the four values have the same semantics internationally;
* whether another governed database field should be used instead of `race_type_raw`.

### Next question

> What race-classification fields already exist alongside `race_type_raw` in the accepted race-level view?


### What we found

`race_type_raw` is complete across all **189,043** governed race occurrences.

There are no null or blank values and only four literal source values:

| `race_type_raw` |   Races |
| --------------- | ------: |
| `Flat`          | 126,391 |
| `Hurdle`        |  35,462 |
| `Chase`         |  22,547 |
| `NH Flat`       |   4,643 |

### Existing governance context

Notebook 16 established that the underlying source field `type` is a race-level attribute and is constant within each governed race occurrence.

Database integration preserves that source value unchanged as `race_type_raw`.

However, Notebook 16 did **not** establish a derived or canonical interpretation for this field. Its governed parsing work instead covered `class`, `pattern`, `rating_band`, `age_band` and `sex_rest`.

Therefore the four labels above are established source facts, but their precise analytical meaning remains the question for this notebook.

### Interpretation

The field is structurally much simpler than many other source fields: it is complete and restricted to four values.

That does not by itself tell us whether those values form a universally valid racing classification.

In particular, we have not yet established whether the same labels are used consistently across jurisdictions or exactly what boundary the source intends between `Flat`, `NH Flat`, `Hurdle` and `Chase`.

### Next question

> Are the four `race_type_raw` values used consistently across jurisdictions, or is their vocabulary jurisdiction-dependent?


### What we found

`race_type_raw` is complete across all **189,043** governed race occurrences.

There are no null or blank values and only four literal source values:

| `race_type_raw` |   Races |
| --------------- | ------: |
| `Flat`          | 126,391 |
| `Hurdle`        |  35,462 |
| `Chase`         |  22,547 |
| `NH Flat`       |   4,643 |

### Existing governance context

Notebook 16 established that the underlying source field `type` is a race-level attribute and is constant within each governed race occurrence.

Database integration preserves that source value unchanged as `race_type_raw`.

However, Notebook 16 did **not** establish a derived or canonical interpretation for this field. Its governed parsing work instead covered `class`, `pattern`, `rating_band`, `age_band` and `sex_rest`.

Therefore the four labels above are established source facts, but their precise analytical meaning remains the question for this notebook.

### Interpretation

The field is structurally much simpler than many other source fields: it is complete and restricted to four values.

That does not by itself tell us whether those values form a universally valid racing classification.

In particular, we have not yet established whether the same labels are used consistently across jurisdictions or exactly what boundary the source intends between `Flat`, `NH Flat`, `Hurdle` and `Chase`.

### Next question

> Are the four `race_type_raw` values used consistently across jurisdictions, or is their vocabulary jurisdiction-dependent?


## 2. Does `race_type_raw` vary by jurisdiction?

The source uses only four `race_type_raw` values globally, but that does not establish that all four labels are used in the same way everywhere.

Jurisdiction is already governed and exposed in the accepted race-level database, so there is no need to reconstruct it from course names.

This step compares the literal `race_type_raw` vocabulary across jurisdictions.

The purpose is to establish:

* which jurisdictions use each raw value;
* whether some jurisdictions use only a subset of the four values;
* whether any value appears confined to particular racing systems;
* whether the apparent four-category vocabulary can reasonably be interpreted globally.

No racing meaning will yet be assigned to the differences.


In [3]:
# Use the existing governed jurisdiction context alongside the untouched
# race_type_raw value. We aggregate in SQLite because the question operates
# at race level and only the small jurisdiction/type summary is needed.
with connect_read_only(DATABASE) as connection:
    jurisdiction_type = pd.read_sql_query(
        f"""
        SELECT
            candidate_jurisdiction,
            race_type_raw,
            COUNT(*) AS races
        FROM {RACE_VIEW}
        GROUP BY
            candidate_jurisdiction,
            race_type_raw
        ORDER BY
            candidate_jurisdiction,
            races DESC,
            race_type_raw
        """,
        connection,
    )

# Calculate the denominator separately for each jurisdiction. This makes the
# percentages describe the composition of racing within that jurisdiction,
# rather than its share of the entire international database.
jurisdiction_type["jurisdiction_races"] = (
    jurisdiction_type
    .groupby("candidate_jurisdiction", dropna=False)["races"]
    .transform("sum")
)

jurisdiction_type["share_pct"] = (
    100
    * jurisdiction_type["races"]
    / jurisdiction_type["jurisdiction_races"]
)

# A count pivot makes absence meaningful: zero means that literal source value
# does not occur in that jurisdiction anywhere in the accepted race population.
count_pivot = (
    jurisdiction_type
    .pivot(
        index="candidate_jurisdiction",
        columns="race_type_raw",
        values="races",
    )
    .fillna(0)
    .astype(int)
)

# Keep the jurisdiction population visible so a tiny jurisdiction is not given
# the same evidential weight as one represented by tens of thousands of races.
count_pivot["Total"] = count_pivot.sum(axis=1)
count_pivot = count_pivot.sort_values("Total", ascending=False)

# Show within-jurisdiction percentages separately from raw counts. We retain
# one decimal place only because greater precision adds no useful information
# at this descriptive stage.
share_pivot = (
    jurisdiction_type
    .pivot(
        index="candidate_jurisdiction",
        columns="race_type_raw",
        values="share_pct",
    )
    .fillna(0)
    .round(1)
)

share_pivot = share_pivot.loc[count_pivot.index]

display(count_pivot)
display(share_pivot)

race_type_raw,Chase,Flat,Hurdle,NH Flat,Total
candidate_jurisdiction,,,,,
Great Britain,15671,70218,22645,3100,111634
Ireland,4833,14763,9667,1520,30783
France,2013,15514,2983,23,20533
Hong Kong,0,7481,0,0,7481
United States,10,6023,72,0,6105
Australia,0,4059,0,0,4059
United Arab Emirates,0,2451,0,0,2451
Japan,7,1546,6,0,1559
Germany,0,731,0,0,731


race_type_raw,Chase,Flat,Hurdle,NH Flat
candidate_jurisdiction,,,,
Great Britain,14.0,62.9,20.3,2.8
Ireland,15.7,48.0,31.4,4.9
France,9.8,75.6,14.5,0.1
Hong Kong,0.0,100.0,0.0,0.0
United States,0.2,98.7,1.2,0.0
Australia,0.0,100.0,0.0,0.0
United Arab Emirates,0.0,100.0,0.0,0.0
Japan,0.4,99.2,0.4,0.0
Germany,0.0,100.0,0.0,0.0


### What we found

The four `race_type_raw` values are **not distributed uniformly across jurisdictions**.

`Flat` is the only source value used in many jurisdictions. Hong Kong, Australia, the United Arab Emirates, Germany, Canada, South Africa, Argentina and numerous smaller populations are recorded entirely as `Flat`.

The other three values are overwhelmingly concentrated in Great Britain, Ireland and France:

* **22,517 of 22,547 `Chase` races (99.87%)** occur in Great Britain, Ireland or France;
* **35,295 of 35,462 `Hurdle` races (99.53%)** occur in those three jurisdictions;
* **all 4,643 `NH Flat` races (100%)** occur in Great Britain, Ireland or France.

`NH Flat` is particularly distinctive. It appears nowhere else in the accepted source population.

There are, however, small but potentially important exceptions outside those three jurisdictions. Examples include:

* United States — 72 `Hurdle`, 10 `Chase`;
* Jersey — 85 `Hurdle`;
* Japan — 6 `Hurdle`, 7 `Chase`;
* Czech Republic — 9 `Chase`;
* Sweden — 3 `Hurdle`;
* Italy — 1 `Hurdle`, 2 `Chase`;
* Switzerland — 1 `Chase`;
* Belgium — 1 `Chase`.

### Interpretation

The source vocabulary is strongly associated with jurisdiction.

This makes a naive global interpretation unsafe. The fact that almost every Australian race is labelled `Flat`, for example, establishes how this source classifies those records; it does not by itself establish a universal definition of flat racing.

Likewise, the small `Hurdle` and `Chase` populations outside Great Britain, Ireland and France should not be treated automatically as errors merely because they are unusual in this dataset. They may represent genuine jumping races, different national terminology mapped into the source vocabulary, or source-classification behaviour.

The concentration itself is evidence, but it does not tell us which explanation is correct.

### What this does not establish

We still do not know:

* what criteria the source uses to assign each of the four labels;
* whether `Hurdle` and `Chase` correspond consistently to the same real-world race forms internationally;
* whether `NH Flat` is a source synonym for a formally defined race category;
* whether the small international `Hurdle` and `Chase` populations are genuine examples or classification anomalies;
* whether `Flat` can safely serve as the complement of all jumping races.

### Next question

The small non-`Flat` populations outside Great Britain, Ireland and France are now the highest-value diagnostic cases.

> Where do those exceptional `Hurdle` and `Chase` classifications occur?


In [4]:
# Focus on the jurisdictional residue rather than inspecting hundreds of
# thousands of ordinary Flat/GB/Irish/French observations. Unusual rows are
# evidence: they may reveal either genuine international jumping races or
# limitations in how the source uses its four race-type labels.
with connect_read_only(DATABASE) as connection:
    other_non_flat_courses = pd.read_sql_query(
        f"""
        SELECT
            candidate_jurisdiction,
            race_type_raw,
            candidate_course_label,
            COUNT(*) AS races
        FROM {RACE_VIEW}
        WHERE race_type_raw <> 'Flat'
          AND candidate_jurisdiction NOT IN (
              'Great Britain',
              'Ireland',
              'France'
          )
        GROUP BY
            candidate_jurisdiction,
            race_type_raw,
            candidate_course_label
        ORDER BY
            races DESC,
            candidate_jurisdiction,
            race_type_raw,
            candidate_course_label
        """,
        connection,
    )

# Preserve the total residue as an audit check. It should equal the difference
# between the global non-Flat counts and the corresponding GB/Ireland/France
# populations observed in the previous table.
residue_races = int(other_non_flat_courses["races"].sum())

expected_residue = (
    (22_547 - 22_517)   # Chase outside GB / Ireland / France
    + (35_462 - 35_295) # Hurdle outside GB / Ireland / France
    + (4_643 - 4_643)   # NH Flat outside GB / Ireland / France
)

assert residue_races == expected_residue == 197, (
    f"Unexpected non-Flat residue: {residue_races:,}"
)

# Course-level aggregation tells us whether the exceptions are concentrated
# at recognisable venues or scattered randomly through the source. That
# distinction will determine whether individual race inspection is worthwhile.
display(other_non_flat_courses)

,candidate_jurisdiction,race_type_raw,candidate_course_label,races
0,Jersey,Hurdle,Les Landes,85
1,United States,Hurdle,Far Hills,27
2,United States,Hurdle,Percy Warner Park,16
3,United States,Hurdle,Saratoga,14
4,Czech Republic,Chase,Pardubice,9
5,United States,Chase,Saratoga,7
6,Japan,Chase,Nakayama,6
7,United States,Hurdle,Belmont Park,6
8,Japan,Hurdle,Nakayama,4
9,Sweden,Hurdle,Bro Park,3


### What we found

The **197 non-`Flat` races outside Great Britain, Ireland and France are strongly concentrated rather than scattered randomly through the source**.

By jurisdiction:

* Jersey — **85 races**;
* United States — **82 races**;
* Japan — **13 races**;
* Czech Republic — **9 races**;
* Sweden — **3 races**;
* Italy — **3 races**;
* Belgium — **1 race**;
* Switzerland — **1 race**.

Jersey and the United States alone account for **167 of 197 races (84.8%)**.

There is also substantial clustering by course. For example:

* all 85 Jersey cases are at `Les Landes`;
* the United States cases are concentrated at a small set of courses including `Far Hills`, `Percy Warner Park`, `Saratoga`, `Belmont Park` and `Great Meadow`;
* 10 of Japan's 13 cases occur at `Nakayama`;
* all 9 Czech cases occur at `Pardubice`;
* all 3 Italian cases occur at `Merano`.

### Interpretation

This pattern does not look like isolated random miscoding.

If the classifications were largely arbitrary source errors, we might expect unusual `Hurdle` and `Chase` values to be dispersed across ordinary courses and jurisdictions. Instead, they repeatedly occur at particular courses and sometimes in substantial numbers.

That makes it increasingly plausible that the source is using `Hurdle` and `Chase` as meaningful race-form classifications outside Britain, Ireland and France as well.

However, course clustering alone is not sufficient to establish what those categories mean. We should first inspect the other source evidence attached to these races before relying on external racing knowledge.

### What this does not establish

We have not yet established:

* what physical obstacles distinguish source `Hurdle` from source `Chase`;
* whether every jurisdiction uses those concepts identically;
* whether the source labels correspond exactly to official local classifications;
* whether any of the 197 races are exceptions or errors despite the overall clustering.

### Next question

> What do the race names and existing classification fields say about these 197 exceptional races?

If `race_type_raw` is meaningful, we would expect the surrounding source evidence to show coherent patterns rather than arbitrary combinations.


In [6]:
# Inspect the other preserved race-classification evidence for the 197-race
# residue. We keep the literal raw fields intact rather than trying to parse
# race names or infer official conditions from title words at this stage.
#
# The reconciled v3 race view carries raw_date directly from the governed v2
# race view, alongside the preserved Notebook 16 classification fields.
with connect_read_only(DATABASE) as connection:
    residue_classification_evidence = pd.read_sql_query(
        f"""
        SELECT
            candidate_jurisdiction,
            candidate_course_label,
            raw_date,
            race_type_raw,
            race_name_raw,
            class_raw,
            pattern_raw,
            rating_band_raw,
            age_band_raw,
            sex_rest_raw
        FROM {RACE_VIEW}
        WHERE race_type_raw <> 'Flat'
          AND candidate_jurisdiction NOT IN (
              'Great Britain',
              'Ireland',
              'France'
          )
        ORDER BY
            candidate_jurisdiction,
            candidate_course_label,
            raw_date,
            race_name_raw
        """,
        connection,
    )

# Protect the exact bounded population established in the previous step.
# If this fails, the detailed evidence no longer describes the same
# non-Flat jurisdictional residue.
assert len(residue_classification_evidence) == 197, (
    "Detailed residue population no longer matches the established 197 races."
)

# At only 197 races, inspect the complete residue rather than an arbitrary
# sample. Rare or unusual combinations may be important semantic evidence.
display(residue_classification_evidence)

,candidate_jurisdiction,candidate_course_label,raw_date,race_type_raw,race_name_raw,class_raw,pattern_raw,rating_band_raw,age_band_raw,sex_rest_raw
0,Belgium,Waregem,2019-08-27,Chase,ING Grand Steeple-Chase des Flandres (Crystal ...,,,,5yo+,
1,Czech Republic,Pardubice,2015-10-11,Chase,Velka Pardubicka S Ceskou Pojistovnou (Chase) ...,,Listed,,6yo+,
2,Czech Republic,Pardubice,2016-10-09,Chase,Velka Pardubicka S Ceskou Pojistovnou (Chase) ...,,Listed,,6yo+,
3,Czech Republic,Pardubice,2017-10-08,Chase,127th Velka Pardubicka S Ceskou Pojistovnou (C...,,Listed,,6yo+,
4,Czech Republic,Pardubice,2018-10-14,Chase,128th Velka Pardubicka S Ceskou Pojistovnou (C...,,Listed,,6yo+,
...,...,...,...,...,...,...,...,...,...,...
192,United States,Saratoga,2024-07-21,Hurdle,A P Smithwick Memorial Handicap (Hurdle) (4yo...,,Grade 1,,4yo+,
193,United States,Saratoga,2024-08-14,Hurdle,Jonathan Sheppard Handicap (Hurdle) (4yo+) (M...,,Grade 1,,4yo+,
194,United States,Saratoga,2025-06-04,Hurdle,Beverly R Steinman Hurdle (Hurdle) Handicap) (...,,Grade 1,,4yo+,
195,United States,Saratoga,2025-07-23,Hurdle,A P Smithwick Memorial Handicap Hurdle (4yo+)...,,Grade 1,,4yo+,


### What we found

The detailed residue provides direct internal evidence that at least many of these unusual international classifications are deliberate rather than accidental.

Visible examples include:

* `Waregem` — source type `Chase`, with `Steeple-Chase` in the race title;
* `Pardubice` — source type `Chase`, with `(Chase)` explicitly present in the title;
* `Saratoga` — source type `Hurdle`, with `(Hurdle)` or `Hurdle` explicitly present in the title.

The same rows also contain coherent surrounding classification information such as `Listed`, `Grade 1`, and age conditions. They do not resemble isolated malformed records.

### Interpretation

This strengthens the evidence that `race_type_raw` is intended to describe a real structural distinction between forms of racing, rather than being an arbitrary four-value label attached independently of the race description.

However, the displayed dataframe is truncated. We should not generalise from the visible examples to all 197 races without measuring the whole bounded population.

Race names also remain supporting evidence rather than authoritative definitions: Notebook 16 already established that words appearing in titles cannot automatically be treated as formal race conditions.

### Next question

> Across all 197 races, how often does the literal race title independently contain the corresponding `Hurdle` or `Chase` wording?


In [7]:
# Use race-name wording only as corroborating source evidence. Notebook 16
# explicitly warns that title phrases are not automatically official race
# conditions, so this step measures agreement rather than defining race type
# from the title.

title_type_agreement = (
    residue_classification_evidence
    .assign(
        title_contains_hurdle=lambda df: (
            df["race_name_raw"]
            .fillna("")
            .str.contains("hurdle", case=False, regex=False)
        ),
        # "chase" also captures forms such as "Steeple-Chase" and
        # "Steeplechase", while retaining the original title untouched.
        title_contains_chase=lambda df: (
            df["race_name_raw"]
            .fillna("")
            .str.contains("chase", case=False, regex=False)
        ),
    )
)

# Count direct lexical agreement between the preserved type value and the
# preserved race title. Failure to match is not treated as a contradiction:
# a race title is not required to repeat its race form explicitly.
title_type_agreement["title_matches_type"] = (
    (
        (title_type_agreement["race_type_raw"] == "Hurdle")
        & title_type_agreement["title_contains_hurdle"]
    )
    |
    (
        (title_type_agreement["race_type_raw"] == "Chase")
        & title_type_agreement["title_contains_chase"]
    )
)

agreement_summary = (
    title_type_agreement
    .groupby("race_type_raw", as_index=False)
    .agg(
        races=("race_type_raw", "size"),
        title_matches=("title_matches_type", "sum"),
    )
)

agreement_summary["title_match_pct"] = (
    100
    * agreement_summary["title_matches"]
    / agreement_summary["races"]
).round(1)

# Keep the unmatched rows visible. They are the informative residue for the
# next decision and must not be silently treated as errors merely because the
# title does not repeat the race-type word.
unmatched_title_cases = title_type_agreement.loc[
    ~title_type_agreement["title_matches_type"],
    [
        "candidate_jurisdiction",
        "candidate_course_label",
        "raw_date",
        "race_type_raw",
        "race_name_raw",
    ],
].copy()

display(agreement_summary)
display(unmatched_title_cases)

,race_type_raw,races,title_matches,title_match_pct
0,Chase,30,30,100.0
1,Hurdle,167,167,100.0


,candidate_jurisdiction,candidate_course_label,raw_date,race_type_raw,race_name_raw


### What we found

Every one of the **197** `Hurdle` or `Chase` races outside Great Britain, Ireland and France has direct corroborating wording in its preserved race title.

* `Chase`: **30 of 30 — 100%**
* `Hurdle`: **167 of 167 — 100%**
* unmatched cases: **0**

This removes the main reason for treating those international observations as suspicious merely because they are rare in the dataset.

### Interpretation

The jurisdictional exceptions are not random classification noise.

For every exceptional case examined, the source independently records the corresponding `Hurdle` or `Chase` wording in `race_name_raw`.

That provides strong source-internal evidence that `race_type_raw` is being used deliberately to distinguish forms of racing across multiple jurisdictions.

It also changes the interpretation of the earlier jurisdiction table. Great Britain, Ireland and France dominate the source's jumping-race population, but the `Hurdle` and `Chase` labels themselves are not confined to those jurisdictions.

We still should not define the categories solely from title text. Race titles are corroborating evidence, not the authoritative source of the classification.

### What this does not establish

We have not yet established whether this same relationship holds across the **entire** `Hurdle`, `Chase` and `NH Flat` population.

In particular, `NH Flat` has not yet been examined semantically.

### Next question

> Across all non-`Flat` races, how consistently does the preserved race title corroborate `race_type_raw`?


In [8]:
# Extend the title-agreement check from the 197 international residue to the
# complete non-Flat population. This tests whether the strong relationship we
# just observed is a general source behaviour rather than a peculiarity of the
# exceptional jurisdictions.
with connect_read_only(DATABASE) as connection:
    non_flat_title_evidence = pd.read_sql_query(
        f"""
        SELECT
            candidate_jurisdiction,
            candidate_course_label,
            raw_date,
            race_type_raw,
            race_name_raw
        FROM {RACE_VIEW}
        WHERE race_type_raw IN (
            'Hurdle',
            'Chase',
            'NH Flat'
        )
        """,
        connection,
    )

# Preserve the previously established global non-Flat population.
expected_non_flat_races = 35_462 + 22_547 + 4_643

assert len(non_flat_title_evidence) == expected_non_flat_races == 62_652, (
    f"Unexpected non-Flat population: {len(non_flat_title_evidence):,}"
)

# These indicators are deliberately lexical only. They test whether the raw
# title contains wording consistent with the raw type; they do not derive or
# replace race_type_raw from the title.
non_flat_title_evidence["title_contains_hurdle"] = (
    non_flat_title_evidence["race_name_raw"]
    .fillna("")
    .str.contains("hurdle", case=False, regex=False)
)

non_flat_title_evidence["title_contains_chase"] = (
    non_flat_title_evidence["race_name_raw"]
    .fillna("")
    .str.contains("chase", case=False, regex=False)
)

# Allow both abbreviated and written-out forms when checking NH Flat titles.
# Any unmatched cases remain evidence for inspection rather than being forced
# into agreement.
race_titles_lower = (
    non_flat_title_evidence["race_name_raw"]
    .fillna("")
    .str.lower()
)

non_flat_title_evidence["title_contains_nh_flat"] = (
    race_titles_lower.str.contains("nh flat", regex=False)
    | race_titles_lower.str.contains("national hunt flat", regex=False)
)

non_flat_title_evidence["title_matches_type"] = (
    (
        (non_flat_title_evidence["race_type_raw"] == "Hurdle")
        & non_flat_title_evidence["title_contains_hurdle"]
    )
    |
    (
        (non_flat_title_evidence["race_type_raw"] == "Chase")
        & non_flat_title_evidence["title_contains_chase"]
    )
    |
    (
        (non_flat_title_evidence["race_type_raw"] == "NH Flat")
        & non_flat_title_evidence["title_contains_nh_flat"]
    )
)

global_agreement_summary = (
    non_flat_title_evidence
    .groupby("race_type_raw", as_index=False)
    .agg(
        races=("race_type_raw", "size"),
        title_matches=("title_matches_type", "sum"),
    )
)

global_agreement_summary["title_match_pct"] = (
    100
    * global_agreement_summary["title_matches"]
    / global_agreement_summary["races"]
).round(1)

# Preserve every non-matching case for inspection. A title need not repeat the
# race form, so non-match means "not lexically corroborated", not "incorrect".
global_unmatched_titles = non_flat_title_evidence.loc[
    ~non_flat_title_evidence["title_matches_type"],
    [
        "candidate_jurisdiction",
        "candidate_course_label",
        "raw_date",
        "race_type_raw",
        "race_name_raw",
    ],
].copy()

display(global_agreement_summary)
display(global_unmatched_titles)

,race_type_raw,races,title_matches,title_match_pct
0,Chase,22547,22547,100.0
1,Hurdle,35462,35462,100.0
2,NH Flat,4643,4534,97.7


,candidate_jurisdiction,candidate_course_label,raw_date,race_type_raw,race_name_raw
1656,Ireland,Fairyhouse,2015-04-05,NH Flat,Tattersalls Ireland George Mernagh Memorial Sa...
3137,Ireland,Killarney,2015-07-13,NH Flat,Killarney Racegoers Club Mares Flat Race
6375,Ireland,Thurles,2016-02-25,NH Flat,Irish Stallion Farms European Breeders Fund Ma...
6997,Ireland,Fairyhouse,2016-03-27,NH Flat,Tattersalls Ireland George Mernagh Memorial Sa...
7864,Ireland,Roscommon,2016-05-09,NH Flat,Kepak Flat Race
...,...,...,...,...,...
58313,Ireland,Gowran Park,2025-10-03,NH Flat,Irish Stallion Farms EBF Mucklemeg Mares Flat ...
60567,Ireland,Leopardstown,2026-02-01,NH Flat,Coolmore N.H. Sires Los Angeles Irish EBF Mare...
60595,Ireland,Leopardstown,2026-02-02,NH Flat,Paddy Power Cheltenham Countdown Podcast (C & ...
62168,Ireland,Punchestown,2026-04-28,NH Flat,Goffs Defender Bumper


### What we found

Across the complete **62,652-race non-`Flat` population**, the source title gives extremely strong independent support for the `race_type_raw` classification:

* `Chase` — **22,547 of 22,547 (100%)** contain `chase` in the race title;
* `Hurdle` — **35,462 of 35,462 (100%)** contain `hurdle`;
* `NH Flat` — **4,534 of 4,643 (97.7%)** matched our initial `NH Flat` / `National Hunt Flat` wording test.

There are **109 apparent `NH Flat` non-matches**.

Inspection of those rows immediately shows that this 109 is not yet evidence of disagreement. Several titles use alternative source wording, including:

* `Flat Race`;
* `Bumper`;
* punctuated forms such as `N.H.`;
* `I.N.H. Flat`.

### Interpretation

For `Hurdle` and `Chase`, the relationship is exceptionally strong: every race classified by the source into either category also contains the corresponding word in its preserved race title.

For `NH Flat`, the lower lexical match rate appears at least partly to be an artefact of our deliberately narrow text search rather than evidence that the source classification is inconsistent.

This is an important distinction. We must not mistake limitations in our diagnostic regex for limitations in the source field.

The next step should therefore investigate the **109-title residue itself**, not conclude that `NH Flat` has a 2.3% disagreement rate.

### Next question

> What wording does the source use in the 109 `NH Flat` titles that did not match our initial literal search?


In [9]:
# Investigate the 109 apparent NH Flat non-matches rather than treating them
# as classification failures. The first lexical test intentionally recognised
# only "NH Flat" and "National Hunt Flat", so alternative source terminology
# and punctuation must be measured explicitly.

nh_flat_unmatched = global_unmatched_titles.loc[
    global_unmatched_titles["race_type_raw"] == "NH Flat"
].copy()

assert len(nh_flat_unmatched) == 109, (
    f"Unexpected NH Flat unmatched population: {len(nh_flat_unmatched):,}"
)

# Normalise only for diagnostic text matching. The original race_name_raw is
# retained unchanged and remains the source evidence.
titles_lower = (
    nh_flat_unmatched["race_name_raw"]
    .fillna("")
    .str.lower()
)

# Test several wording families visible in the source residue.
#
# These flags are descriptive only: finding "bumper" or "flat race" does not
# yet authorise either phrase as a formal definition of NH Flat.
nh_flat_unmatched["mentions_flat_race"] = (
    titles_lower.str.contains(r"\bflat\s+race\b", regex=True)
)

nh_flat_unmatched["mentions_bumper"] = (
    titles_lower.str.contains(r"\bbumper\b", regex=True)
)

# Allow punctuation and spacing in abbreviations such as:
# N.H. Flat
# I.N.H. Flat
nh_flat_unmatched["mentions_punctuated_nh_flat"] = (
    titles_lower.str.contains(
        r"\b(?:i\s*\.\s*)?n\s*\.\s*h\s*\.\s*flat\b",
        regex=True,
    )
)

nh_flat_unmatched["mentions_national_hunt"] = (
    titles_lower.str.contains(
        r"\bnational\s+hunt\b",
        regex=True,
    )
)

# Summarise how much of the 109-race residue is explained by each observed
# wording family. Categories may overlap, so these counts are not intended
# to sum to 109.
wording_summary = pd.DataFrame(
    {
        "wording_family": [
            "Flat Race",
            "Bumper",
            "punctuated N.H. / I.N.H. Flat",
            "National Hunt",
        ],
        "races": [
            int(nh_flat_unmatched["mentions_flat_race"].sum()),
            int(nh_flat_unmatched["mentions_bumper"].sum()),
            int(nh_flat_unmatched["mentions_punctuated_nh_flat"].sum()),
            int(nh_flat_unmatched["mentions_national_hunt"].sum()),
        ],
    }
)

# Identify only the remaining cases for which none of these obvious source
# wording families provides lexical corroboration.
known_wording = (
    nh_flat_unmatched[
        [
            "mentions_flat_race",
            "mentions_bumper",
            "mentions_punctuated_nh_flat",
            "mentions_national_hunt",
        ]
    ]
    .any(axis=1)
)

remaining_unexplained = nh_flat_unmatched.loc[
    ~known_wording,
    [
        "candidate_jurisdiction",
        "candidate_course_label",
        "raw_date",
        "race_name_raw",
    ],
].copy()

# Jurisdiction matters because terminology may reflect local racing vocabulary.
jurisdiction_summary = (
    nh_flat_unmatched
    .groupby("candidate_jurisdiction", as_index=False)
    .size()
    .rename(columns={"size": "races"})
    .sort_values("races", ascending=False)
)

display(wording_summary)
display(jurisdiction_summary)
display(remaining_unexplained)

,wording_family,races
0,Flat Race,98
1,Bumper,11
2,punctuated N.H. / I.N.H. Flat,24
3,National Hunt,0


,candidate_jurisdiction,races
0,Ireland,109


,candidate_jurisdiction,candidate_course_label,raw_date,race_name_raw


### What we found

The 109 initially unmatched `NH Flat` titles are entirely confined to Ireland.

Every one is accounted for by alternative source wording:

* **98** contain `Flat Race`;
* **11** contain `Bumper`;
* **24** also contain punctuated `N.H.` / `I.N.H. Flat` wording;
* **0** cases remain outside the tested wording families.

The categories overlap, so these counts are not additive.

### Interpretation

The earlier 97.7% literal title-match rate for `NH Flat` was primarily a consequence of using an overly narrow lexical test.

This does not yet establish that `Flat Race`, `Bumper`, `N.H. Flat` and `I.N.H. Flat` are formally equivalent regulatory terms. It establishes only that the apparently unmatched source records contain coherent alternative wording rather than unexplained or obviously contradictory titles.

The source-internal evidence for `race_type_raw` is therefore now very strong:

* all `Hurdle` races contain `hurdle` in their titles;
* all `Chase` races contain `chase`;
* the apparent `NH Flat` title residue is completely explained by recurring alternative terminology.

However, an earlier database investigation recorded **eight explicit `NH Flat` / type conflicts** and deliberately deferred them to race-type/classification governance.

Those eight cases are now the relevant unresolved edge case.

### Next question

> Are the eight previously deferred `NH Flat` conflicts the races where the source simultaneously records `NH Flat` and an explicit all-weather course marker?


## 3. Return to the Great Britain study blocker

The wider profiling above established useful background about `race_type_raw`, but it went beyond the question that caused this database investigation.

The original issue arose in:

`studies/jurisdictions/great_britain/01_governance_and_structure.ipynb`

That study grouped Great Britain races into provisional course-date meetings and examined whether a broader meeting-level distinction between **Flat racing** and **National Hunt racing** was defensible.

Almost all meetings appeared compatible with that distinction, but **25 course-date meetings contained both**:

* at least one race labelled `Flat`; and
* at least one race labelled `Hurdle`, `Chase` or `NH Flat`.

Those 25 apparent mixed-code meetings were unexpected.

Before treating them as genuine features of British racing, we need to establish whether their underlying `race_type_raw` assignments are credible.

The bounded question for this investigation is therefore:

> **Do the race-level `race_type_raw` values inside the 25 apparent mixed Flat/National Hunt Great Britain meetings appear correctly assigned?**

We will first reconstruct the exact 25-meeting population directly from Database v3 and inspect every constituent race.

At this stage:

* `Flat` means only the literal source value `Flat`;
* National Hunt means only the observed source values `Hurdle`, `Chase` and `NH Flat`;
* these groupings are diagnostic labels for reproducing the study result, not yet a new governed taxonomy;
* race-title wording will be supporting evidence, not an automatic replacement for `race_type_raw`.


In [10]:
# Reconstruct the exact population that blocked the Great Britain structure
# study. A provisional meeting is defined exactly as it was there: one
# candidate course label on one raw calendar date.
#
# The Flat/National Hunt grouping below is used only to identify the 25
# unexpected mixed meetings. It does not create a new governed race-type field.
with connect_read_only(DATABASE) as connection:
    mixed_gb_races = pd.read_sql_query(
        f"""
        WITH gb_races AS (
            SELECT
                source_race_occurrence_id,
                raw_date,
                raw_off,
                raw_course,
                candidate_course_label,
                race_type_raw,
                race_name_raw
            FROM {RACE_VIEW}
            WHERE candidate_jurisdiction = 'Great Britain'
        ),
        mixed_meetings AS (
            SELECT
                raw_date,
                candidate_course_label
            FROM gb_races
            GROUP BY
                raw_date,
                candidate_course_label
            HAVING
                SUM(
                    CASE
                        WHEN race_type_raw = 'Flat' THEN 1
                        ELSE 0
                    END
                ) > 0
                AND
                SUM(
                    CASE
                        WHEN race_type_raw IN ('Hurdle', 'Chase', 'NH Flat')
                        THEN 1
                        ELSE 0
                    END
                ) > 0
        )
        SELECT
            r.source_race_occurrence_id,
            r.raw_date,
            r.candidate_course_label,
            r.raw_course,
            r.raw_off,
            r.race_type_raw,
            r.race_name_raw
        FROM gb_races AS r
        JOIN mixed_meetings AS m
          ON m.raw_date = r.raw_date
         AND m.candidate_course_label = r.candidate_course_label
        ORDER BY
            r.raw_date,
            r.candidate_course_label,
            r.raw_off,
            r.source_race_occurrence_id
        """,
        connection,
    )

# Protect the study result we are trying to investigate. If this is not 25,
# this notebook has failed to reproduce the original blocker and we should
# stop rather than analyse a different population.
mixed_meeting_count = (
    mixed_gb_races[
        ["raw_date", "candidate_course_label"]
    ]
    .drop_duplicates()
    .shape[0]
)

assert mixed_meeting_count == 25, (
    f"Expected 25 mixed GB meetings, found {mixed_meeting_count:,}"
)

# Make the number of races and the exact race-type composition of each meeting
# visible before judging whether any individual classification looks wrong.
mixed_meeting_summary = (
    mixed_gb_races
    .groupby(
        ["raw_date", "candidate_course_label"],
        as_index=False,
    )
    .agg(
        races=("source_race_occurrence_id", "size"),
        flat_races=("race_type_raw", lambda s: int((s == "Flat").sum())),
        hurdle_races=("race_type_raw", lambda s: int((s == "Hurdle").sum())),
        chase_races=("race_type_raw", lambda s: int((s == "Chase").sum())),
        nh_flat_races=("race_type_raw", lambda s: int((s == "NH Flat").sum())),
    )
)

display(mixed_meeting_summary)
display(
    mixed_gb_races[
        [
            "raw_date",
            "candidate_course_label",
            "raw_off",
            "race_type_raw",
            "race_name_raw",
        ]
    ]
)

,raw_date,candidate_course_label,races,flat_races,hurdle_races,chase_races,nh_flat_races
0,2015-02-13,Sandown,7,1,4,2,0
1,2015-03-06,Sandown,6,1,4,1,0
2,2015-05-09,Haydock,7,3,2,2,0
3,2015-05-14,Fontwell,7,1,0,6,0
4,2016-01-30,Doncaster,8,2,3,3,0
5,2016-03-11,Sandown,6,1,4,1,0
6,2016-05-07,Haydock,7,3,2,2,0
7,2017-05-13,Haydock,8,4,2,2,0
8,2017-08-04,Bath,6,5,0,1,0
9,2018-05-12,Haydock,8,4,2,2,0


,raw_date,candidate_course_label,raw_off,race_type_raw,race_name_raw
0,2015-02-13,Sandown,1:30,Hurdle,EstatesDirect.com The 0% Commission Agent Cond...
1,2015-02-13,Sandown,2:00,Chase,Alanbrooke Handicap Chase
2,2015-02-13,Sandown,2:30,Hurdle,Weatherbys GSB Jane Seymour Mares Novices Hurd...
3,2015-02-13,Sandown,3:05,Flat,Royal Artillery Gold Cup (Chase For Military A...
4,2015-02-13,Sandown,3:40,Hurdle,David Lindon & Co Novices Hurdle
...,...,...,...,...,...
175,2026-05-09,Haydock,13:48,NH Flat,Pertemps Network Junior National Hunt Flat Rac...
176,2026-05-09,Haydock,14:30,Flat,Pertemps Newton Handicap
177,2026-05-09,Haydock,15:05,Flat,Pertemps Network Handicap
178,2026-05-09,Haydock,15:40,Flat,Pertemps Network Spring Trophy Stakes


### What we found

The reconstruction reproduces the original study blocker exactly:

* **25** apparent mixed Flat/National Hunt Great Britain meetings;
* **180 races** in those meetings;
* **73 races** are labelled `Flat`.

The detailed race output immediately reveals at least one explicit contradiction.

At Sandown on **13 February 2015**, the 3:05 race is stored as:

* `race_type_raw = Flat`

but its preserved race title is:

* `Royal Artillery Gold Cup (Chase For Military Amateur Riders)`

That is strong source-internal evidence that the `Flat` assignment is wrong for this race.

### Interpretation

This materially changes the meeting-level result.

The 25 apparent mixed-code meetings cannot yet be treated as genuine examples of Flat and National Hunt racing taking place within the same course-date fixture.

At least one is already demonstrably capable of being produced by an incorrect `race_type_raw` value.

The next step is therefore not to analyse meeting structure further. It is to inspect **all 73 races labelled `Flat` within these 25 meetings** for the same kind of contradiction.

This remains a diagnostic exercise. Race-title wording will identify strong candidates for incorrect source types; it will not silently overwrite the database.

### Next question

> How many of the 73 `Flat` assignments inside the 25 apparent mixed meetings are contradicted by explicit National Hunt wording in their race titles?


In [11]:
# The mixed-meeting result can be created by a single incorrect Flat label.
# Therefore inspect every Flat-labelled race in the 25-meeting population
# before assuming that any meeting genuinely combines racing codes.
mixed_flat_races = mixed_gb_races.loc[
    mixed_gb_races["race_type_raw"] == "Flat"
].copy()

# Protect the population visible in the previous meeting summary.
assert len(mixed_flat_races) == 73, (
    f"Expected 73 Flat-labelled races, found {len(mixed_flat_races):,}"
)

# Normalise title text only for diagnostic matching. race_name_raw remains
# unchanged and is the preserved evidence used for later manual inspection.
flat_titles = (
    mixed_flat_races["race_name_raw"]
    .fillna("")
    .str.lower()
)

# Explicit Hurdle or Chase wording is a direct contradiction to a Flat type.
# National Hunt Flat / NH Flat / Bumper wording is also relevant because those
# races have their own observed source category rather than ordinary Flat.
mixed_flat_races["mentions_hurdle"] = (
    flat_titles.str.contains("hurdle", regex=False)
)

mixed_flat_races["mentions_chase"] = (
    flat_titles.str.contains("chase", regex=False)
)

mixed_flat_races["mentions_nh_flat"] = (
    flat_titles.str.contains("national hunt flat", regex=False)
    | flat_titles.str.contains("nh flat", regex=False)
    | flat_titles.str.contains("n.h. flat", regex=False)
    | flat_titles.str.contains(r"\bbumper\b", regex=True)
)

mixed_flat_races["explicit_nh_contradiction"] = (
    mixed_flat_races[
        [
            "mentions_hurdle",
            "mentions_chase",
            "mentions_nh_flat",
        ]
    ]
    .any(axis=1)
)

# Summarise the scale of the problem before looking at individual rows.
flat_contradiction_summary = pd.DataFrame(
    {
        "flat_labelled_races": [len(mixed_flat_races)],
        "explicit_nh_title_contradictions": [
            int(mixed_flat_races["explicit_nh_contradiction"].sum())
        ],
        "remaining_flat_without_explicit_nh_wording": [
            int((~mixed_flat_races["explicit_nh_contradiction"]).sum())
        ],
    }
)

# Show every contradictory row. At this bounded size there is no reason to
# sample: each one may explain one of the 25 apparent mixed meetings.
flat_type_contradictions = mixed_flat_races.loc[
    mixed_flat_races["explicit_nh_contradiction"],
    [
        "raw_date",
        "candidate_course_label",
        "raw_off",
        "race_type_raw",
        "race_name_raw",
        "mentions_hurdle",
        "mentions_chase",
        "mentions_nh_flat",
    ],
].copy()

display(flat_contradiction_summary)
display(flat_type_contradictions)

,flat_labelled_races,explicit_nh_title_contradictions,remaining_flat_without_explicit_nh_wording
0,73,9,64


,raw_date,candidate_course_label,raw_off,race_type_raw,race_name_raw,mentions_hurdle,mentions_chase,mentions_nh_flat
3,2015-02-13,Sandown,3:05,Flat,Royal Artillery Gold Cup (Chase For Military A...,False,True,False
9,2015-03-06,Sandown,3:25,Flat,Grand Military Gold Cup (Chase For Military Am...,False,True,False
33,2016-01-30,Doncaster,3:50,Flat,Stevie Bows 50th Birthday Celebration British ...,False,False,True
34,2016-01-30,Doncaster,4:25,Flat,Stevie Bows 50th Birthday Celebration British ...,False,False,True
37,2016-03-11,Sandown,3:15,Flat,Grand Military Gold Cup (Chase for Military Am...,False,True,False
76,2018-06-08,Stratford,9:00,Flat,Irish Thoroughbred Marketing Champion Point-To...,False,False,True
91,2019-05-31,Stratford,8:50,Flat,Irish Thoroughbred Marketing Champion Point-To...,False,False,True
106,2021-05-28,Stratford,8:40,Flat,Irish Thoroughbred Marketing Champion Point-To...,False,False,True
154,2024-07-12,Chepstow,6:25,Flat,Cruises Chase Handicap,False,True,False


### What we found

Of the **73 races labelled `Flat`** within the 25 apparent mixed-code Great Britain meetings, **9 contain explicit National Hunt terminology in their own race titles**.

Those nine occur across **8 distinct meetings**:

* Sandown — 13 February 2015;
* Sandown — 6 March 2015;
* Doncaster — 30 January 2016;
* Sandown — 11 March 2016;
* Stratford — 8 June 2018;
* Stratford — 31 May 2019;
* Stratford — 28 May 2021;
* Chepstow — 12 July 2024.

Several already appear strongly contradictory. For example, the Sandown military races are labelled `Flat` despite containing `Chase` explicitly in their titles.

However, lexical matching alone is not enough to promote all nine immediately to source errors. A word such as `Chase` can potentially occur as part of a proper name, sponsorship phrase or other title wording.

We therefore need to inspect the complete, untruncated titles of these nine races before deciding which are genuine contradictions.

### Next question

> What exactly do the full titles of the nine candidate contradictions say?


In [14]:
# Reconstruct the exact population that blocked the Great Britain structure
# study. A provisional meeting is defined exactly as it was there: one
# candidate course label on one source calendar date.
#
# The Flat/National Hunt grouping below is used only to identify the 25
# unexpected mixed meetings. It does not create a new governed race-type field.
#
# Race ordering and presentation use the governed advertised/scheduled
# course-local timestamp. Raw source `off`, UK-facing time and UTC are not used
# here because this study does not require those alternative representations.
with connect_read_only(DATABASE) as connection:
    mixed_gb_races = pd.read_sql_query(
        f"""
        WITH gb_races AS (
            SELECT
                source_race_occurrence_id,
                raw_date,
                raw_course,
                candidate_course_label,
                advertised_start_course_local,
                temporal_resolution_status,
                race_type_raw,
                race_name_raw
            FROM {RACE_VIEW}
            WHERE candidate_jurisdiction = 'Great Britain'
        ),
        mixed_meetings AS (
            SELECT
                raw_date,
                candidate_course_label
            FROM gb_races
            GROUP BY
                raw_date,
                candidate_course_label
            HAVING
                SUM(
                    CASE
                        WHEN race_type_raw = 'Flat' THEN 1
                        ELSE 0
                    END
                ) > 0
                AND
                SUM(
                    CASE
                        WHEN race_type_raw IN (
                            'Hurdle',
                            'Chase',
                            'NH Flat'
                        )
                        THEN 1
                        ELSE 0
                    END
                ) > 0
        )
        SELECT
            r.source_race_occurrence_id,
            r.raw_date,
            r.candidate_course_label,
            r.raw_course,
            r.advertised_start_course_local,
            r.temporal_resolution_status,
            r.race_type_raw,
            r.race_name_raw
        FROM gb_races AS r
        JOIN mixed_meetings AS m
          ON m.raw_date = r.raw_date
         AND m.candidate_course_label = r.candidate_course_label
        ORDER BY
            r.raw_date,
            r.candidate_course_label,
            r.advertised_start_course_local,
            r.source_race_occurrence_id
        """,
        connection,
    )

# Protect the exact study blocker. If this does not reproduce the original
# 25 course-date meetings, stop rather than analyse a different population.
mixed_meeting_count = (
    mixed_gb_races[
        ["raw_date", "candidate_course_label"]
    ]
    .drop_duplicates()
    .shape[0]
)

assert mixed_meeting_count == 25, (
    f"Expected 25 mixed GB meetings, found {mixed_meeting_count:,}"
)

# Summarise each apparent mixed meeting before deciding whether its individual
# race-type assignments are credible.
mixed_meeting_summary = (
    mixed_gb_races
    .groupby(
        ["raw_date", "candidate_course_label"],
        as_index=False,
    )
    .agg(
        races=("source_race_occurrence_id", "size"),
        flat_races=("race_type_raw", lambda s: int((s == "Flat").sum())),
        hurdle_races=("race_type_raw", lambda s: int((s == "Hurdle").sum())),
        chase_races=("race_type_raw", lambda s: int((s == "Chase").sum())),
        nh_flat_races=("race_type_raw", lambda s: int((s == "NH Flat").sum())),
    )
)

# Keep the governed full timestamp in the analytical dataframe. Create a
# presentation-only local clock value so the notebook reads like a racecard.
mixed_gb_races_display = mixed_gb_races.copy()

mixed_gb_races_display["local_off_time"] = (
    mixed_gb_races_display["advertised_start_course_local"]
    .str.slice(11, 16)
)

display(mixed_meeting_summary)

display(
    mixed_gb_races_display[
        [
            "raw_date",
            "candidate_course_label",
            "local_off_time",
            "temporal_resolution_status",
            "race_type_raw",
            "race_name_raw",
        ]
    ]
)

,raw_date,candidate_course_label,races,flat_races,hurdle_races,chase_races,nh_flat_races
0,2015-02-13,Sandown,7,1,4,2,0
1,2015-03-06,Sandown,6,1,4,1,0
2,2015-05-09,Haydock,7,3,2,2,0
3,2015-05-14,Fontwell,7,1,0,6,0
4,2016-01-30,Doncaster,8,2,3,3,0
5,2016-03-11,Sandown,6,1,4,1,0
6,2016-05-07,Haydock,7,3,2,2,0
7,2017-05-13,Haydock,8,4,2,2,0
8,2017-08-04,Bath,6,5,0,1,0
9,2018-05-12,Haydock,8,4,2,2,0


,raw_date,candidate_course_label,local_off_time,temporal_resolution_status,race_type_raw,race_name_raw
0,2015-02-13,Sandown,13:30,resolved,Hurdle,EstatesDirect.com The 0% Commission Agent Cond...
1,2015-02-13,Sandown,14:00,resolved,Chase,Alanbrooke Handicap Chase
2,2015-02-13,Sandown,14:30,resolved,Hurdle,Weatherbys GSB Jane Seymour Mares Novices Hurd...
3,2015-02-13,Sandown,15:05,resolved,Flat,Royal Artillery Gold Cup (Chase For Military A...
4,2015-02-13,Sandown,15:40,resolved,Hurdle,David Lindon & Co Novices Hurdle
...,...,...,...,...,...,...
175,2026-05-09,Haydock,13:48,resolved,NH Flat,Pertemps Network Junior National Hunt Flat Rac...
176,2026-05-09,Haydock,14:30,resolved,Flat,Pertemps Newton Handicap
177,2026-05-09,Haydock,15:05,resolved,Flat,Pertemps Network Handicap
178,2026-05-09,Haydock,15:40,resolved,Flat,Pertemps Network Spring Trophy Stakes


### What we found

The original study blocker is reproduced exactly.

Across the Great Britain population, **25 course-date meetings** contain at least one race labelled `Flat` and at least one race labelled `Hurdle`, `Chase` or `NH Flat`.

Those meetings contain **180 races** in total.

The detailed output immediately shows that the apparent mixing cannot simply be accepted as a genuine feature of British racing. For example, Sandown on 13 February 2015 contains a race labelled `Flat` whose title is `Royal Artillery Gold Cup (Chase For Military Amateur Riders)`.

### Interpretation

The 25 meetings are therefore an anomaly population requiring race-level verification.

At least one apparent Flat/National Hunt mixture can already be explained by a suspicious `race_type_raw` assignment rather than by a genuinely mixed racing programme.

The next step is to inspect every `Flat`-labelled race within these 25 meetings for explicit contradictory National Hunt wording, while preserving the raw source values unchanged.

### Next question

> How many of the `Flat` assignments within these 25 meetings are contradicted by their own race titles?


## 4. Inspect the nine candidate `Flat` contradictions

The diagnostic search identified **9 of the 73 `Flat`-labelled races** whose race titles contain explicit National Hunt terminology.

That lexical result is evidence for inspection, not yet proof that every one of the nine is incorrectly classified.

For example, a word such as `Chase` could theoretically occur within a proper name rather than describe the form of the race.

We therefore inspect the complete preserved race titles for all nine candidates before assigning any interpretation.

### Question

> Do the complete race titles show that these nine `Flat` assignments are genuinely inconsistent with the races being described?


In [15]:
# Review every candidate rather than sampling because there are only nine and
# each potentially explains one of the 25 apparent mixed-code meetings.
flat_type_contradictions_review = (
    flat_type_contradictions[
        [
            "raw_date",
            "candidate_course_label",
            "advertised_start_course_local",
            "race_name_raw",
            "mentions_hurdle",
            "mentions_chase",
            "mentions_nh_flat",
        ]
    ]
    .copy()
)

# Create a presentation-only local off time from the governed course-local
# advertised timestamp. The full governed timestamp remains unchanged.
flat_type_contradictions_review["local_off_time"] = (
    flat_type_contradictions_review[
        "advertised_start_course_local"
    ]
    .str.slice(11, 16)
)

flat_type_contradictions_review = (
    flat_type_contradictions_review[
        [
            "raw_date",
            "candidate_course_label",
            "local_off_time",
            "race_name_raw",
            "mentions_hurdle",
            "mentions_chase",
            "mentions_nh_flat",
        ]
    ]
    .sort_values(
        [
            "raw_date",
            "candidate_course_label",
            "local_off_time",
        ]
    )
    .reset_index(drop=True)
)

# Suppress pandas truncation so the decision is based on the complete preserved
# title rather than an abbreviated notebook display.
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None,
    "display.width", 220,
):
    display(flat_type_contradictions_review)

KeyError: "['advertised_start_course_local'] not in index"

In [16]:
# Rebuild the nine candidate Flat contradictions from the current mixed-meeting
# dataframe so this review does not depend on a stale intermediate dataframe
# created before course-local race time was added.
mixed_flat_races = mixed_gb_races.loc[
    mixed_gb_races["race_type_raw"] == "Flat"
].copy()

# Protect the population established in the previous analytical step.
assert len(mixed_flat_races) == 73, (
    f"Expected 73 Flat-labelled races, found {len(mixed_flat_races):,}"
)

# Normalise title text only for diagnostic matching. The preserved
# race_name_raw value itself is not modified.
flat_titles = (
    mixed_flat_races["race_name_raw"]
    .fillna("")
    .str.lower()
)

mixed_flat_races["mentions_hurdle"] = (
    flat_titles.str.contains("hurdle", regex=False)
)

mixed_flat_races["mentions_chase"] = (
    flat_titles.str.contains("chase", regex=False)
)

mixed_flat_races["mentions_nh_flat"] = (
    flat_titles.str.contains("national hunt flat", regex=False)
    | flat_titles.str.contains("nh flat", regex=False)
    | flat_titles.str.contains("n.h. flat", regex=False)
    | flat_titles.str.contains(r"\bbumper\b", regex=True)
)

mixed_flat_races["explicit_nh_contradiction"] = (
    mixed_flat_races[
        [
            "mentions_hurdle",
            "mentions_chase",
            "mentions_nh_flat",
        ]
    ]
    .any(axis=1)
)

# Retain every flagged race. There are only nine, so the evidence should be
# reviewed exhaustively rather than sampled.
flat_type_contradictions_review = mixed_flat_races.loc[
    mixed_flat_races["explicit_nh_contradiction"],
    [
        "raw_date",
        "candidate_course_label",
        "advertised_start_course_local",
        "race_name_raw",
        "mentions_hurdle",
        "mentions_chase",
        "mentions_nh_flat",
    ],
].copy()

assert len(flat_type_contradictions_review) == 9, (
    "Expected 9 candidate contradictions, found "
    f"{len(flat_type_contradictions_review):,}"
)

# Convert the governed course-local timestamp to a normal racecard-style clock
# value for presentation only. The analytical timestamp remains unchanged.
flat_type_contradictions_review["local_off_time"] = (
    flat_type_contradictions_review[
        "advertised_start_course_local"
    ]
    .str.slice(11, 16)
)

flat_type_contradictions_review = (
    flat_type_contradictions_review[
        [
            "raw_date",
            "candidate_course_label",
            "local_off_time",
            "race_name_raw",
            "mentions_hurdle",
            "mentions_chase",
            "mentions_nh_flat",
        ]
    ]
    .sort_values(
        [
            "raw_date",
            "candidate_course_label",
            "local_off_time",
        ]
    )
    .reset_index(drop=True)
)

# Show complete titles because the classification decision depends on the exact
# wording, not on pandas' abbreviated display.
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None,
    "display.width", 220,
):
    display(flat_type_contradictions_review)

,raw_date,candidate_course_label,local_off_time,race_name_raw,mentions_hurdle,mentions_chase,mentions_nh_flat
0,2015-02-13,Sandown,15:05,Royal Artillery Gold Cup (Chase For Military Amateur Riders)(Supported By Morgan Advanced Materials),False,True,False
1,2015-03-06,Sandown,15:25,Grand Military Gold Cup (Chase For Military Amateur Riders) (Sponsored By The Military Mutual),False,True,False
2,2016-01-30,Doncaster,15:50,Stevie Bows 50th Birthday Celebration British Stallions EBF Mares Standard Open NH Flat (Div I),False,False,True
3,2016-01-30,Doncaster,16:25,Stevie Bows 50th Birthday Celebration British Stallions EBF Mares Standard Open NH Flat (Div II),False,False,True
4,2016-03-11,Sandown,15:15,Grand Military Gold Cup (Chase for Military Amateur Riders) (Sponsored by The Military Mutual),False,True,False
5,2018-06-08,Stratford,NaN,Irish Thoroughbred Marketing Champion Point-To-Point Bumper (A Standard NHF Race) (Amateur Riders),False,False,True
6,2019-05-31,Stratford,NaN,Irish Thoroughbred Marketing Champion Point-To-Point Bumper (A Standard NHF Race) (Amateur Riders),False,False,True
7,2021-05-28,Stratford,NaN,Irish Thoroughbred Marketing Champion Point-To-Point Bumper (Standard NHF Race) (GBB Race),False,False,True
8,2024-07-12,Chepstow,18:25,Cruises Chase Handicap,False,True,False


### What we found

Of the **73 races labelled `Flat`** within the 25 apparent mixed Flat/National Hunt meetings, **9 race titles contained explicit National Hunt-looking terminology**:

* 3 contained `Chase`;
* 6 contained `NH Flat`, `NHF` or `Bumper` terminology.

The title diagnostic was deliberately treated only as a way of finding candidates. A word appearing in a race name is not sufficient evidence by itself to change the source classification.

We therefore externally checked **all nine races** against published result information.

### External verification

The checks established that **8 of the 9 `Flat` assignments are wrong**.

The following races were externally identified as National Hunt races despite the source storing `race_type_raw = Flat`:

* **Sandown, 13 February 2015 — Royal Artillery Gold Cup:** Chase;
* **Sandown, 6 March 2015 — Grand Military Gold Cup:** Chase;
* **Doncaster, 30 January 2016 — Stevie Bows 50th Birthday Celebration... Div I:** NH Flat;
* **Doncaster, 30 January 2016 — Stevie Bows 50th Birthday Celebration... Div II:** NH Flat;
* **Sandown, 11 March 2016 — Grand Military Gold Cup:** Chase;
* **Stratford, 8 June 2018 — Champion Point-To-Point Bumper:** NH Flat;
* **Stratford, 31 May 2019 — Champion Point-To-Point Bumper:** NH Flat;
* **Stratford, 28 May 2021 — Champion Point-To-Point Bumper:** NH Flat.

The ninth candidate demonstrates why title matching cannot itself be used as a correction rule:

* **Chepstow, 12 July 2024 — Cruises Chase Handicap** was externally confirmed to be an ordinary **Flat handicap**. `Chase` is part of the race name rather than a description of the racing code.

Therefore:

> **8 of the 9 title-based candidates are confirmed source race-type errors; 1 is a lexical false positive.**

### Additional temporal evidence

The external checks also resolved the missing course-local times for the three Stratford races whose governed advertised times were previously unresolved:

* **8 June 2018:** advertised at **21:00**; reported actual off approximately **21:01**;
* **31 May 2019:** **20:50**;
* **28 May 2021:** **20:40**.

These temporal findings are separate from the race-type corrections. They should not be conflated with the classification issue, but they provide additional externally established facts for the next governed database update.

### Interpretation

This establishes that at least part of the original **25 apparent mixed Flat/National Hunt meetings** result is caused by incorrect source `race_type_raw` assignments.

In particular, several races stored as `Flat` are demonstrably Chases or National Hunt Flat races.

The exercise also establishes an important methodological safeguard: **race-title wording can identify cases worth checking, but it cannot safely determine race type automatically**. `Cruises Chase Handicap` is a concrete counterexample.

### What this does not establish

We cannot yet conclude how many of the 25 meetings were genuinely mixed Flat/National Hunt programmes.

So far we have tested only one direction of possible classification error:

> races stored as `Flat` that may actually have been National Hunt races.

The reverse problem may also exist. A race stored as `Hurdle`, `Chase` or `NH Flat` inside an otherwise Flat meeting could itself be wrongly classified.

### Next question

> Do any of the `Hurdle`, `Chase` or `NH Flat` assignments within the 25 apparent mixed meetings also appear to be incorrectly assigned?


## 5. Inspect the opposite direction of the apparent mixing

The first diagnostic found several races labelled `Flat` whose titles suggested that they were actually National Hunt races.

That only tests one possible error direction.

An apparent mixed meeting could also be created by the reverse problem: a race labelled `Hurdle`, `Chase` or `NH Flat` within an otherwise predominantly Flat card might itself be incorrectly classified.

We therefore inspect every National Hunt-labelled race occurring in a meeting where `Flat` is the majority race type.

### Question

> What are the `Hurdle`, `Chase` and `NH Flat` races inside Flat-majority meetings, and do their complete race titles support those assignments?


In [17]:
# Measure each apparent mixed meeting at the broader Flat-versus-National-Hunt
# level. This grouping is diagnostic only and does not create a governed code.
meeting_code_balance = (
    mixed_gb_races
    .groupby(
        ["raw_date", "candidate_course_label"],
        as_index=False,
    )
    .agg(
        flat_races=("race_type_raw", lambda s: int((s == "Flat").sum())),
        nh_races=(
            "race_type_raw",
            lambda s: int(
                s.isin(["Hurdle", "Chase", "NH Flat"]).sum()
            ),
        ),
    )
)

# Restrict this step to meetings where Flat races outnumber all National Hunt
# labels combined. These are the clearest places to look for the reverse error:
# an isolated or minority NH label inside a predominantly Flat programme.
flat_majority_meetings = meeting_code_balance.loc[
    meeting_code_balance["flat_races"]
    > meeting_code_balance["nh_races"]
].copy()

flat_majority_nh_races = (
    mixed_gb_races
    .merge(
        flat_majority_meetings[
            [
                "raw_date",
                "candidate_course_label",
                "flat_races",
                "nh_races",
            ]
        ],
        on=["raw_date", "candidate_course_label"],
        how="inner",
        validate="many_to_one",
    )
    .loc[
        lambda df: df["race_type_raw"].isin(
            ["Hurdle", "Chase", "NH Flat"]
        )
    ]
    .copy()
)

# Use the governed course-local timestamp for race context. Where temporal
# governance remains unresolved, make that explicit rather than substituting
# the raw source clock.
flat_majority_nh_races["local_off_time"] = (
    flat_majority_nh_races[
        "advertised_start_course_local"
    ]
    .str.slice(11, 16)
    .fillna("UNRESOLVED")
)

flat_majority_nh_races = (
    flat_majority_nh_races
    .sort_values(
        [
            "raw_date",
            "candidate_course_label",
            "advertised_start_course_local",
            "source_race_occurrence_id",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

print(
    "Flat-majority apparent mixed meetings:",
    len(flat_majority_meetings),
)
print(
    "NH-labelled races requiring inspection:",
    len(flat_majority_nh_races),
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None,
    "display.width", 240,
):
    display(
        flat_majority_nh_races[
            [
                "raw_date",
                "candidate_course_label",
                "flat_races",
                "nh_races",
                "local_off_time",
                "temporal_resolution_status",
                "race_type_raw",
                "race_name_raw",
            ]
        ]
    )

Flat-majority apparent mixed meetings: 8
NH-labelled races requiring inspection: 18


,raw_date,candidate_course_label,flat_races,nh_races,local_off_time,temporal_resolution_status,race_type_raw,race_name_raw
0,2017-08-04,Bath,5,1,19:40,resolved,Chase,Kingstone Press Wild Berry Chase Handicap (Bath Summer Stayers Series Qualifier)
1,2022-05-07,Haydock,5,3,14:25,resolved,Hurdle,Pertemps Network Long Distance Handicap Hurdle (GBB Race)
2,2022-05-07,Haydock,5,3,15:00,resolved,Hurdle,Pertemps Network Swinton Handicap Hurdle (GBB Race)
3,2022-05-07,Haydock,5,3,16:10,resolved,NH Flat,Pertemps Network Open National Hunt Flat Race (Category 1 Elimination) (GBB Race)
4,2023-05-13,Haydock,5,3,13:35,resolved,Hurdle,Pertemps Network Long Distance Handicap Hurdle (GBB Race)
5,2023-05-13,Haydock,5,3,15:15,resolved,Hurdle,Pertemps Network Swinton Handicap Hurdle (Premier Handicap) (GBB Race)
6,2023-05-13,Haydock,5,3,17:00,resolved,NH Flat,Pertemps Network Open National Hunt Flat Race (Category 1 Elimination) (GBB Race)
7,2024-05-11,Haydock,4,3,13:35,resolved,Hurdle,Pertemps Network Long Distance Handicap Hurdle (GBB Race)
8,2024-05-11,Haydock,4,3,15:15,resolved,Hurdle,Pertemps Network Swinton Handicap Hurdle (Premier Handicap) (GBB Race)
9,2024-05-11,Haydock,4,3,16:25,resolved,NH Flat,Pertemps Network Junior Open National Hunt Flat Race (Category 1 Elimination) (GBB Race)


### What we found

The reverse-direction check identified **18 races labelled `Hurdle`, `Chase` or `NH Flat` within 8 Flat-majority apparent mixed meetings**.

The pattern was highly concentrated:

* **15 races** belonged to recurring Haydock meetings between 2022 and 2026;
* the remaining **3 races** were isolated National Hunt labels at Bath, Chepstow and Epsom.

That distinction matters because the two groups have very different explanations.

### Haydock: genuine mixed Flat and National Hunt programmes

The Haydock races are not classification anomalies.

Published results confirm that these meetings genuinely combined Flat and National Hunt racing.

For example, the 7 May 2022 card contained:

* the **Long Distance Handicap Hurdle**, run over hurdles;
* the **Swinton Handicap Hurdle**, also explicitly run over hurdles;
* an **Open National Hunt Flat Race**;
* alongside ordinary Flat races later on the same card.

The same structure continued in later years. Published Haydock results explicitly distinguish the **Hurdle course** from the **Flat course** on the same meeting in both 2023 and 2024, while recording the Swinton and Long Distance races as genuine hurdles.

The 2025 card was likewise described as a mixed jumps-and-Flat card and included two hurdles and a Junior National Hunt Flat race before the Flat programme.

The 2026 published results again separately report **Jumps** and **Flat** going and include the Long Distance Hurdle, Swinton Hurdle and Junior National Hunt Flat Race.

The Haydock `Hurdle` and `NH Flat` assignments therefore appear correct. These meetings are genuine examples of Flat and National Hunt racing occurring on the same programme.

### Bath: confirmed reverse classification error

Bath on 4 August 2017 contains five races labelled `Flat` and one race labelled `Chase`:

> `Kingstone Press Wild Berry Chase Handicap`

External result evidence shows that this was **not a steeplechase**.

It was a **2m1f Flat handicap on turf**, with ordinary Flat-race conditions and runners. Independent result sources explicitly classify it as Flat.

Therefore:

> **Bath 4 August 2017: `race_type_raw = Chase` is wrong; the race was Flat.**

Here, as with `Cruises Chase Handicap`, the word `Chase` occurs in the race name without describing the racing code.

### Chepstow: confirmed reverse classification error

The isolated `Chase` race at Chepstow on 12 July 2024 was:

> `Hullabaloos Chase Handicap`

Published results identify it as a **Class 6 Flat handicap over 1m4f on turf**. Timeform explicitly records its race type as `Flat`, and other result sources give ordinary Flat-race conditions and draw numbers.

Therefore:

> **Chepstow 12 July 2024: `race_type_raw = Chase` is wrong; the race was Flat.**

### Epsom: confirmed reverse classification error

The isolated `Hurdle` race at Epsom on 12 September 2024 was:

> `No Hurdles With Emplas Jump Jockeys Derby Handicap (For Professional Jump Jockeys)`

Despite the words `Hurdles` and `Jump Jockeys`, the race itself was a **Flat handicap over approximately 1m4f on turf**.

The unusual condition was that professional jump jockeys rode in it; it was not a hurdle race. Published results explicitly describe it as Flat.

Therefore:

> **Epsom 12 September 2024: `race_type_raw = Hurdle` is wrong; the race was Flat.**

### Interpretation

This reverse-direction check finds **3 further confirmed `race_type_raw` errors**:

| Date        | Course   | Raw type | Verified type |
| ----------- | -------- | -------- | ------------- |
| 4 Aug 2017  | Bath     | `Chase`  | **Flat**      |
| 12 Jul 2024 | Chepstow | `Chase`  | **Flat**      |
| 12 Sep 2024 | Epsom    | `Hurdle` | **Flat**      |

Combined with the previous investigation, we have now externally confirmed **11 incorrect `race_type_raw` assignments** within the original 25 apparent mixed meetings:

* **8** races stored as `Flat` that were actually Chase or NH Flat races;
* **3** races stored as `Chase` or `Hurdle` that were actually Flat races.

The Haydock cases also establish something important in the opposite direction: **mixed Flat/National Hunt meetings are real**, so the correct solution is not simply to assume that every mixed meeting represents bad data.

### Methodological finding

Race names are clearly implicated in several source classification errors.

Examples now include:

* `Royal Artillery Gold Cup (Chase...)` stored as Flat;
* `Grand Military Gold Cup (Chase...)` stored as Flat;
* explicit `NH Flat` races stored as Flat;
* `Wild Berry Chase Handicap` stored as Chase even though it was Flat;
* `Hullabaloos Chase Handicap` stored as Chase even though it was Flat;
* `No Hurdles With Emplas Jump Jockeys Derby` stored as Hurdle even though it was Flat.

This is consistent with some classifications having been derived incorrectly from title wording, but **we have not established the source's classification algorithm and should not claim that mechanism as fact**.

What we can establish is that **title text cannot safely be converted into race type by simple keyword rules in either direction**.

### Next question

We should now apply the **11 externally established corrections analytically, without altering the immutable raw values**, and ask:

> **After those confirmed race-level errors are corrected, how many of the original 25 apparent mixed Flat/National Hunt meetings remain genuinely mixed?**


## 6. Recompute the 25 meetings using only externally verified corrections

We have now externally established **11 incorrect `race_type_raw` assignments** within the original 25 apparent mixed meetings.

To determine how much of the original result survives those confirmed errors, we temporarily apply only those 11 verified corrections to the diagnostic dataframe.

This does **not** modify Database v3, overwrite `race_type_raw`, or create a new governed database field. The corrected value exists only inside this investigation so that we can measure the consequence of the external verification.

The original raw value remains preserved alongside it.

### Question

> After applying only the 11 externally verified race-type corrections, how many of the original 25 apparent mixed Flat/National Hunt meetings remain mixed?


In [18]:
# Record only the 11 race-type corrections that have been established by
# external verification during this investigation. These are diagnostic
# amendments for measuring the effect on the original 25-meeting result;
# they do not modify Database v3 or overwrite race_type_raw.
verified_race_type_corrections = pd.DataFrame(
    [
        {
            "raw_date": "2015-02-13",
            "candidate_course_label": "Sandown",
            "race_name_raw": (
                "Royal Artillery Gold Cup (Chase For Military Amateur Riders)"
                "(Supported By Morgan Advanced Materials)"
            ),
            "expected_raw_type": "Flat",
            "verified_race_type": "Chase",
        },
        {
            "raw_date": "2015-03-06",
            "candidate_course_label": "Sandown",
            "race_name_raw": (
                "Grand Military Gold Cup (Chase For Military Amateur Riders) "
                "(Sponsored By The Military Mutual)"
            ),
            "expected_raw_type": "Flat",
            "verified_race_type": "Chase",
        },
        {
            "raw_date": "2016-01-30",
            "candidate_course_label": "Doncaster",
            "race_name_raw": (
                "Stevie Bows 50th Birthday Celebration British Stallions EBF "
                "Mares Standard Open NH Flat (Div I)"
            ),
            "expected_raw_type": "Flat",
            "verified_race_type": "NH Flat",
        },
        {
            "raw_date": "2016-01-30",
            "candidate_course_label": "Doncaster",
            "race_name_raw": (
                "Stevie Bows 50th Birthday Celebration British Stallions EBF "
                "Mares Standard Open NH Flat (Div II)"
            ),
            "expected_raw_type": "Flat",
            "verified_race_type": "NH Flat",
        },
        {
            "raw_date": "2016-03-11",
            "candidate_course_label": "Sandown",
            "race_name_raw": (
                "Grand Military Gold Cup (Chase for Military Amateur Riders) "
                "(Sponsored by The Military Mutual)"
            ),
            "expected_raw_type": "Flat",
            "verified_race_type": "Chase",
        },
        {
            "raw_date": "2017-08-04",
            "candidate_course_label": "Bath",
            "race_name_raw": (
                "Kingstone Press Wild Berry Chase Handicap "
                "(Bath Summer Stayers Series Qualifier)"
            ),
            "expected_raw_type": "Chase",
            "verified_race_type": "Flat",
        },
        {
            "raw_date": "2018-06-08",
            "candidate_course_label": "Stratford",
            "race_name_raw": (
                "Irish Thoroughbred Marketing Champion Point-To-Point Bumper "
                "(A Standard NHF Race) (Amateur Riders)"
            ),
            "expected_raw_type": "Flat",
            "verified_race_type": "NH Flat",
        },
        {
            "raw_date": "2019-05-31",
            "candidate_course_label": "Stratford",
            "race_name_raw": (
                "Irish Thoroughbred Marketing Champion Point-To-Point Bumper "
                "(A Standard NHF Race) (Amateur Riders)"
            ),
            "expected_raw_type": "Flat",
            "verified_race_type": "NH Flat",
        },
        {
            "raw_date": "2021-05-28",
            "candidate_course_label": "Stratford",
            "race_name_raw": (
                "Irish Thoroughbred Marketing Champion Point-To-Point Bumper "
                "(Standard NHF Race) (GBB Race)"
            ),
            "expected_raw_type": "Flat",
            "verified_race_type": "NH Flat",
        },
        {
            "raw_date": "2024-07-12",
            "candidate_course_label": "Chepstow",
            "race_name_raw": "Hullabaloos Chase Handicap",
            "expected_raw_type": "Chase",
            "verified_race_type": "Flat",
        },
        {
            "raw_date": "2024-09-12",
            "candidate_course_label": "Epsom",
            "race_name_raw": (
                "No Hurdles With Emplas Jump Jockeys Derby Handicap "
                "(For Professional Jump Jockeys)"
            ),
            "expected_raw_type": "Hurdle",
            "verified_race_type": "Flat",
        },
    ]
)

assert len(verified_race_type_corrections) == 11

# Work on a copy so the immutable source classification remains visible and
# available for comparison throughout the investigation.
mixed_gb_races_verified = mixed_gb_races.copy()
mixed_gb_races_verified["race_type_verified"] = (
    mixed_gb_races_verified["race_type_raw"]
)

# Apply each verified correction only after proving that the identifying
# date + course + full race title resolves to exactly one race and that its
# current raw type is the value that external evidence contradicted.
for correction in verified_race_type_corrections.itertuples(index=False):
    mask = (
        (mixed_gb_races_verified["raw_date"] == correction.raw_date)
        & (
            mixed_gb_races_verified["candidate_course_label"]
            == correction.candidate_course_label
        )
        & (
            mixed_gb_races_verified["race_name_raw"]
            == correction.race_name_raw
        )
    )

    matching_rows = mixed_gb_races_verified.loc[mask]

    assert len(matching_rows) == 1, (
        "Verified correction did not resolve to exactly one race: "
        f"{correction.raw_date} | "
        f"{correction.candidate_course_label} | "
        f"{correction.race_name_raw}"
    )

    actual_raw_type = matching_rows["race_type_raw"].iloc[0]

    assert actual_raw_type == correction.expected_raw_type, (
        "Raw race type no longer matches the externally checked value: "
        f"expected {correction.expected_raw_type!r}, "
        f"found {actual_raw_type!r}"
    )

    mixed_gb_races_verified.loc[
        mask, "race_type_verified"
    ] = correction.verified_race_type


# Confirm that exactly 11 race rows changed and nothing else was silently
# altered by the diagnostic reconciliation.
changed_rows = (
    mixed_gb_races_verified["race_type_raw"]
    != mixed_gb_races_verified["race_type_verified"]
)

assert int(changed_rows.sum()) == 11


# Recalculate Flat-versus-National-Hunt composition for the same original
# 25 course-date meetings using the externally verified diagnostic type.
verified_meeting_summary = (
    mixed_gb_races_verified
    .groupby(
        ["raw_date", "candidate_course_label"],
        as_index=False,
    )
    .agg(
        races=("source_race_occurrence_id", "size"),
        flat_races=(
            "race_type_verified",
            lambda s: int((s == "Flat").sum()),
        ),
        hurdle_races=(
            "race_type_verified",
            lambda s: int((s == "Hurdle").sum()),
        ),
        chase_races=(
            "race_type_verified",
            lambda s: int((s == "Chase").sum()),
        ),
        nh_flat_races=(
            "race_type_verified",
            lambda s: int((s == "NH Flat").sum()),
        ),
    )
)

# A meeting remains mixed only if it still contains at least one verified
# Flat race and at least one verified Hurdle/Chase/NH Flat race.
verified_meeting_summary["remains_mixed"] = (
    (verified_meeting_summary["flat_races"] > 0)
    & (
        (
            verified_meeting_summary["hurdle_races"]
            + verified_meeting_summary["chase_races"]
            + verified_meeting_summary["nh_flat_races"]
        )
        > 0
    )
)

remaining_mixed_meetings = verified_meeting_summary.loc[
    verified_meeting_summary["remains_mixed"]
].copy()

resolved_as_single_code = verified_meeting_summary.loc[
    ~verified_meeting_summary["remains_mixed"]
].copy()

print(
    "Original apparent mixed meetings:",
    len(verified_meeting_summary),
)
print(
    "Remain mixed after 11 verified corrections:",
    len(remaining_mixed_meetings),
)
print(
    "No longer mixed after verified corrections:",
    len(resolved_as_single_code),
)

display(verified_meeting_summary)

Original apparent mixed meetings: 25
Remain mixed after 11 verified corrections: 15
No longer mixed after verified corrections: 10


,raw_date,candidate_course_label,races,flat_races,hurdle_races,chase_races,nh_flat_races,remains_mixed
0,2015-02-13,Sandown,7,0,4,3,0,False
1,2015-03-06,Sandown,6,0,4,2,0,False
2,2015-05-09,Haydock,7,3,2,2,0,True
3,2015-05-14,Fontwell,7,1,0,6,0,True
4,2016-01-30,Doncaster,8,0,3,3,2,False
5,2016-03-11,Sandown,6,0,4,2,0,False
6,2016-05-07,Haydock,7,3,2,2,0,True
7,2017-05-13,Haydock,8,4,2,2,0,True
8,2017-08-04,Bath,6,6,0,0,0,False
9,2018-05-12,Haydock,8,4,2,2,0,True


### What we found

Applying only the **11 externally verified race-type corrections** materially changes the original result.

The original diagnostic identified:

* **25 apparent mixed Flat/National Hunt meetings**.

After correcting only races whose type has been independently established:

* **15 meetings remain mixed**;
* **10 meetings are no longer mixed**.

The 10 meetings removed from the mixed population are:

* Sandown — 13 February 2015;
* Sandown — 6 March 2015;
* Doncaster — 30 January 2016;
* Sandown — 11 March 2016;
* Bath — 4 August 2017;
* Stratford — 8 June 2018;
* Stratford — 31 May 2019;
* Stratford — 28 May 2021;
* Chepstow — 12 July 2024;
* Epsom — 12 September 2024.

These disappear for exactly the reasons established by the external checks. Doncaster contains two incorrect `Flat` assignments, so **11 corrected races remove 10 apparent mixed meetings**.

### Interpretation

This is a substantial correction to the study result.

**40% of the original apparent mixed meetings — 10 of 25 — were artefacts of confirmed `race_type_raw` errors.**

The original 25-meeting count therefore cannot be treated as a reliable description of British racing structure.

However, the corrections do **not** eliminate mixed meetings altogether.

The remaining **15 meetings** consist of:

* **11 Haydock meetings**, spanning 2015–2026;
* Fontwell — 14 May 2015;
* Sandown — 11 March 2023;
* Southwell — 26 June 2023;
* Market Rasen — 6 August 2023.

Five of the later Haydock meetings — 2022 through 2026 — have already been externally checked and shown to be genuine mixed Flat/National Hunt programmes.

That leaves **10 remaining meetings requiring verification** before we can state the final prevalence of genuine mixed meetings:

* six earlier Haydock meetings from 2015–2021;
* Fontwell 2015;
* Sandown 2023;
* Southwell 2023;
* Market Rasen 2023.

### What this establishes

We can now say with confidence that:

> The source contains material race-type classification errors capable of creating entirely false mixed-code meetings.

We can also say that:

> Genuine mixed Flat/National Hunt programmes do exist, because externally verified Haydock examples survive the correction process.

What we cannot yet say is whether **all 15 remaining meetings** are genuine.

### Next question

> Of the 15 meetings that remain mixed after the confirmed corrections, which are genuine mixed Flat/National Hunt programmes and which, if any, contain further race-type errors?


## 7. Inspect the 10 remaining unverified mixed meetings

After applying the 11 externally verified race-type corrections, **15 meetings remain mixed**.

Five of those — Haydock from 2022 through 2026 — have already been externally verified as genuine mixed Flat/National Hunt programmes.

That leaves **10 meetings whose apparent mixing has not yet been verified**:

* Haydock — 9 May 2015;
* Fontwell — 14 May 2015;
* Haydock — 7 May 2016;
* Haydock — 13 May 2017;
* Haydock — 12 May 2018;
* Haydock — 11 May 2019;
* Haydock — 8 May 2021;
* Sandown — 11 March 2023;
* Southwell — 26 June 2023;
* Market Rasen — 6 August 2023.

Before performing further external checks, inspect the complete race programmes for those meetings using the diagnostically corrected race type.

### Question

> What exactly were the races making up the 10 remaining unverified mixed meetings?


In [19]:
# The five Haydock meetings from 2022-2026 have already been externally
# established as genuine mixed Flat/National Hunt programmes. Restrict the next
# inspection to the ten surviving meetings whose mixed status remains unverified.
already_verified_mixed_meetings = {
    ("2022-05-07", "Haydock"),
    ("2023-05-13", "Haydock"),
    ("2024-05-11", "Haydock"),
    ("2025-05-10", "Haydock"),
    ("2026-05-09", "Haydock"),
}

remaining_unverified_meetings = remaining_mixed_meetings.loc[
    ~remaining_mixed_meetings.apply(
        lambda row: (
            row["raw_date"],
            row["candidate_course_label"],
        )
        in already_verified_mixed_meetings,
        axis=1,
    )
].copy()

assert len(remaining_unverified_meetings) == 10, (
    "Expected 10 remaining unverified mixed meetings, found "
    f"{len(remaining_unverified_meetings):,}"
)

# Join back to the race-level diagnostic population so we can inspect every race
# in each surviving meeting. Use race_type_verified rather than race_type_raw
# because the 11 externally established corrections must remain in force during
# this investigation.
remaining_unverified_races = (
    mixed_gb_races_verified
    .merge(
        remaining_unverified_meetings[
            ["raw_date", "candidate_course_label"]
        ],
        on=["raw_date", "candidate_course_label"],
        how="inner",
        validate="many_to_one",
    )
    .copy()
)

# Present the governed course-local advertised time as a normal racecard clock.
# If temporal governance remains unresolved, expose that state explicitly rather
# than silently falling back to raw source `off`.
remaining_unverified_races["local_off_time"] = (
    remaining_unverified_races[
        "advertised_start_course_local"
    ]
    .str.slice(11, 16)
    .fillna("UNRESOLVED")
)

remaining_unverified_races = (
    remaining_unverified_races
    .sort_values(
        [
            "raw_date",
            "candidate_course_label",
            "advertised_start_course_local",
            "source_race_occurrence_id",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

print(
    "Remaining unverified mixed meetings:",
    len(remaining_unverified_meetings),
)
print(
    "Race rows requiring programme inspection:",
    len(remaining_unverified_races),
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None,
    "display.width", 260,
):
    display(
        remaining_unverified_races[
            [
                "raw_date",
                "candidate_course_label",
                "local_off_time",
                "temporal_resolution_status",
                "race_type_raw",
                "race_type_verified",
                "race_name_raw",
            ]
        ]
    )

Remaining unverified mixed meetings: 10
Race rows requiring programme inspection: 75


,raw_date,candidate_course_label,local_off_time,temporal_resolution_status,race_type_raw,race_type_verified,race_name_raw
0,2015-05-09,Haydock,13:45,resolved,Hurdle,Hurdle,Pertemps Network Long Distance Handicap Hurdle
1,2015-05-09,Haydock,14:15,resolved,Flat,Flat,Pertemps Network Handicap
2,2015-05-09,Haydock,14:50,resolved,Flat,Flat,Pertemps Network Conditions Stakes
3,2015-05-09,Haydock,15:25,resolved,Hurdle,Hurdle,Pertemps Network Handicap Hurdle (Registered as The Swinton Hurdle)
4,2015-05-09,Haydock,16:00,resolved,Flat,Flat,Pertemps Network Spring Trophy Stakes ()
5,2015-05-09,Haydock,16:35,resolved,Chase,Chase,Pertemps Network Intermediate Handicap Chase
6,2015-05-09,Haydock,17:05,resolved,Chase,Chase,Pertemps Network Handicap Chase
7,2015-05-14,Fontwell,17:05,resolved,Flat,Flat,racebets.com - Claim Your £50 Welcome Bonus! Novices Hunters Chs (Guy Peate Memorial Chall Trphy)
8,2015-05-14,Fontwell,17:40,resolved,Chase,Chase,Call Star Sports On 08000 521321 Novices Hunters Chase
9,2015-05-14,Fontwell,18:15,resolved,Chase,Chase,starsportsbet.co.uk Ladies Open Hunters Chase (for the Stuart Adamson Memorial Trophy)


### What we found

Inspecting the complete programmes reduced the remaining uncertainty to two distinct cases.

#### The six earlier Haydock meetings are genuine mixed programmes

The Haydock programmes from **2015, 2016, 2017, 2018, 2019 and 2021** genuinely contained both Flat and jumping races.

This is not merely inferred from the source titles. Published results independently show both forms of racing on the same cards.

For example:

* **9 May 2015** included a Long Distance Handicap Hurdle and Intermediate Handicap Chase, while the 14:15 Pertemps Network Handicap was a Flat race.
* **7 May 2016** included the Swinton Handicap Hurdle alongside the 7f Spring Trophy Stakes.
* **13 May 2017** included Flat racing, hurdles and chases; the published result information explicitly distinguishes the Flat and chase/hurdle going.
* **12 May 2018** included the Flat Stayers' Handicap, Swinton Handicap Hurdle and subsequent chases.
* **11 May 2019** explicitly reports separate `Flat course` and `Jumps course` going and contains both forms on the same card.
* **8 May 2021** likewise reports separate Flat and Jumps course conditions and contains Flat races, hurdles and chases.

Together with the already verified Haydock meetings from 2022–2026, this establishes that the recurring Haydock May programme is a genuine mixed Flat/National Hunt fixture rather than a classification artefact.

### Four further race-type errors

The other four surviving non-Haydock meetings each contain one race stored as `Flat` whose published result establishes a National Hunt type.

#### Fontwell — 14 May 2015

The 17:05 race is stored as:

`race_type_raw = Flat`

but its title is:

`racebets.com - Claim Your £50 Welcome Bonus! Novices Hunters Chs`

Published results identify it as a **Novices' Hunters' Chase**, run over approximately 3m2½f.

Verified correction:

> `Flat` → **Chase**

This means the entire Fontwell card was a chase programme rather than a genuine mixed Flat/Chase meeting.

#### Sandown — 11 March 2023

The 13:50 race is stored as:

`race_type_raw = Flat`

but is:

`European Breeders' Fund Betfair "National Hunt" Novices' Handicap Hdl Final`

Published results identify it as a **2m4f hurdle race**, including seven hurdles with two omitted.

Verified correction:

> `Flat` → **Hurdle**

The apparent Sandown mixed meeting therefore disappears.

#### Southwell — 26 June 2023

The 15:00 race is stored as:

`race_type_raw = Flat`

but its title explicitly describes a:

`Mares Open NHF Race`

Published results confirm it as an **Open National Hunt Flat Race** over approximately two miles.

Verified correction:

> `Flat` → **NH Flat**

The Southwell card is therefore a National Hunt programme, not a genuine mixed Flat/National Hunt meeting.

#### Market Rasen — 6 August 2023

The 17:22 race is stored as:

`race_type_raw = Flat`

but is titled:

`Mares Open NHF Race`

External results explicitly classify it as **NH Flat**.

Verified correction:

> `Flat` → **NH Flat**

The Market Rasen card is therefore also a National Hunt programme rather than a genuine mixed meeting.

### Interpretation

These checks establish **4 additional incorrect `race_type_raw` assignments**.

Combined with the previous 11 externally verified errors, the investigation has now identified:

> **15 confirmed incorrect race-type assignments within the original 25 apparent mixed meetings.**

Those errors account for **14 of the 25 apparent mixed meetings**.

The remaining **11 meetings are all Haydock meetings**, and external evidence now supports them as genuine mixed Flat/National Hunt programmes.

The original result therefore changes from:

> **25 apparent mixed meetings**

to:

> **11 externally supported genuine mixed meetings, all at Haydock**

and:

> **14 false mixed meetings created by source race-type errors.**

That means **56% of the original apparent mixed-meeting population was spurious**.

### What this establishes

The broader Flat/National Hunt meeting distinction remains analytically useful, but `race_type_raw` cannot be accepted blindly even within Great Britain.

The source errors are uncommon in the overall race population, but they are highly consequential for this particular structural question because a **single wrongly classified race is enough to turn an otherwise single-code meeting into an apparent mixed meeting**.

This also explains why inspecting anomalous meetings rather than simply accepting the aggregate count was necessary.

### Next question

> After applying all 15 externally verified race-type corrections, does the original 25-meeting anomaly population reduce exactly to the 11 genuine Haydock mixed meetings?


## 8. Apply all 15 externally verified race-type corrections

The investigation has now externally verified **15 incorrect `race_type_raw` assignments** within the original 25 apparent mixed Flat/National Hunt meetings.

The first reconciliation applied 11 of those corrections. Four additional errors have since been established at:

* Fontwell — 14 May 2015;
* Sandown — 11 March 2023;
* Southwell — 26 June 2023;
* Market Rasen — 6 August 2023.

We now apply those four additional verified corrections to the existing diagnostic dataframe and recompute the same original 25 meetings.

This remains an investigative reconciliation only. `race_type_raw` is preserved unchanged and Database v3 is not modified.

### Question

> After applying all 15 externally verified corrections, do the original 25 apparent mixed meetings reduce exactly to the 11 externally supported Haydock mixed meetings?


In [20]:
# Add only the four further corrections established by external verification
# after inspecting the ten remaining unverified mixed programmes.
additional_verified_race_type_corrections = pd.DataFrame(
    [
        {
            "raw_date": "2015-05-14",
            "candidate_course_label": "Fontwell",
            "race_name_raw": (
                "racebets.com - Claim Your £50 Welcome Bonus! Novices Hunters Chs "
                "(Guy Peate Memorial Chall Trphy)"
            ),
            "expected_raw_type": "Flat",
            "verified_race_type": "Chase",
        },
        {
            "raw_date": "2023-03-11",
            "candidate_course_label": "Sandown",
            "race_name_raw": (
                "European Breeders Fund Betfair National Hunt Novices Handicap "
                "Hdl Final (Premier Handicap) (GBB)"
            ),
            "expected_raw_type": "Flat",
            "verified_race_type": "Hurdle",
        },
        {
            "raw_date": "2023-06-26",
            "candidate_course_label": "Southwell",
            "race_name_raw": (
                "Prestige Safety e-learning @ prestigesafetyservices.com "
                "Mares Open NHF Race (Cat 1 Elim) (GBB)"
            ),
            "expected_raw_type": "Flat",
            "verified_race_type": "NH Flat",
        },
        {
            "raw_date": "2023-08-06",
            "candidate_course_label": "Market Rasen",
            "race_name_raw": (
                "Streets Chartered Accountants & Streets Bloodstock Mares Open "
                "NHF Race (Rider Restricted)(C1)(GBB)"
            ),
            "expected_raw_type": "Flat",
            "verified_race_type": "NH Flat",
        },
    ]
)

assert len(additional_verified_race_type_corrections) == 4


# Continue from the dataframe that already contains the first 11 externally
# verified corrections. Preserve race_type_raw and amend only race_type_verified.
mixed_gb_races_verified_15 = mixed_gb_races_verified.copy()

for correction in additional_verified_race_type_corrections.itertuples(index=False):
    mask = (
        (mixed_gb_races_verified_15["raw_date"] == correction.raw_date)
        & (
            mixed_gb_races_verified_15["candidate_course_label"]
            == correction.candidate_course_label
        )
        & (
            mixed_gb_races_verified_15["race_name_raw"]
            == correction.race_name_raw
        )
    )

    matching_rows = mixed_gb_races_verified_15.loc[mask]

    # Fail closed if the external evidence cannot be linked to exactly one
    # source race in the original anomaly population.
    assert len(matching_rows) == 1, (
        "Additional verified correction did not resolve to exactly one race: "
        f"{correction.raw_date} | "
        f"{correction.candidate_course_label} | "
        f"{correction.race_name_raw}"
    )

    actual_raw_type = matching_rows["race_type_raw"].iloc[0]

    # Protect against accidentally applying an external correction to a row
    # whose preserved raw classification is not the one that was verified.
    assert actual_raw_type == correction.expected_raw_type, (
        "Raw race type no longer matches the externally checked value: "
        f"expected {correction.expected_raw_type!r}, "
        f"found {actual_raw_type!r}"
    )

    mixed_gb_races_verified_15.loc[
        mask,
        "race_type_verified",
    ] = correction.verified_race_type


# There should now be exactly 15 race rows where the externally supported
# diagnostic value differs from the preserved source race_type_raw value.
changed_rows_15 = (
    mixed_gb_races_verified_15["race_type_raw"]
    != mixed_gb_races_verified_15["race_type_verified"]
)

assert int(changed_rows_15.sum()) == 15


# Recompute the Flat / Hurdle / Chase / NH Flat composition for exactly the
# same original 25 course-date meetings. This isolates the consequence of the
# verified corrections without changing the original anomaly population.
verified_15_meeting_summary = (
    mixed_gb_races_verified_15
    .groupby(
        ["raw_date", "candidate_course_label"],
        as_index=False,
    )
    .agg(
        races=("source_race_occurrence_id", "size"),
        flat_races=(
            "race_type_verified",
            lambda s: int((s == "Flat").sum()),
        ),
        hurdle_races=(
            "race_type_verified",
            lambda s: int((s == "Hurdle").sum()),
        ),
        chase_races=(
            "race_type_verified",
            lambda s: int((s == "Chase").sum()),
        ),
        nh_flat_races=(
            "race_type_verified",
            lambda s: int((s == "NH Flat").sum()),
        ),
    )
)

verified_15_meeting_summary["remains_mixed"] = (
    (verified_15_meeting_summary["flat_races"] > 0)
    & (
        (
            verified_15_meeting_summary["hurdle_races"]
            + verified_15_meeting_summary["chase_races"]
            + verified_15_meeting_summary["nh_flat_races"]
        )
        > 0
    )
)

final_mixed_meetings = verified_15_meeting_summary.loc[
    verified_15_meeting_summary["remains_mixed"]
].copy()

final_single_code_meetings = verified_15_meeting_summary.loc[
    ~verified_15_meeting_summary["remains_mixed"]
].copy()


# The investigation predicts that every surviving mixed meeting should be at
# Haydock. Assert that explicitly rather than relying on visual inspection.
assert len(final_mixed_meetings) == 11, (
    f"Expected 11 verified mixed meetings, found {len(final_mixed_meetings):,}"
)

assert set(final_mixed_meetings["candidate_course_label"]) == {"Haydock"}, (
    "A non-Haydock meeting still remains mixed after the 15 verified corrections"
)

print("Original apparent mixed meetings:", len(verified_15_meeting_summary))
print(
    "Remain mixed after 15 verified corrections:",
    len(final_mixed_meetings),
)
print(
    "No longer mixed after verified corrections:",
    len(final_single_code_meetings),
)

display(final_mixed_meetings)

Original apparent mixed meetings: 25
Remain mixed after 15 verified corrections: 11
No longer mixed after verified corrections: 14


,raw_date,candidate_course_label,races,flat_races,hurdle_races,chase_races,nh_flat_races,remains_mixed
2,2015-05-09,Haydock,7,3,2,2,0,True
6,2016-05-07,Haydock,7,3,2,2,0,True
7,2017-05-13,Haydock,8,4,2,2,0,True
9,2018-05-12,Haydock,8,4,2,2,0,True
11,2019-05-11,Haydock,8,4,2,2,0,True
13,2021-05-08,Haydock,8,4,2,2,0,True
15,2022-05-07,Haydock,8,5,2,0,1,True
17,2023-05-13,Haydock,8,5,2,0,1,True
20,2024-05-11,Haydock,7,4,2,0,1,True
23,2025-05-10,Haydock,7,4,2,0,1,True


### What we found

After applying all **15 externally verified race-type corrections** to the original anomaly population:

* original apparent mixed meetings: **25**;
* meetings no longer mixed: **14**;
* meetings that remain mixed: **11**.

Every surviving mixed meeting is at **Haydock**.

The surviving meetings are:

* 9 May 2015;
* 7 May 2016;
* 13 May 2017;
* 12 May 2018;
* 11 May 2019;
* 8 May 2021;
* 7 May 2022;
* 13 May 2023;
* 11 May 2024;
* 10 May 2025;
* 9 May 2026.

External result checks support these Haydock programmes as genuine combinations of Flat and National Hunt racing.

### Final interpretation

The original reader-facing study found **25 apparent mixed Flat/National Hunt Great Britain meetings**.

That result was materially distorted by incorrect source race classifications.

External verification established **15 incorrect `race_type_raw` assignments** within the 25-meeting anomaly population.

Those errors created **14 entirely false mixed meetings**.

After correcting only externally established errors:

> **11 genuine mixed Flat/National Hunt meetings remain, all at Haydock.**

Therefore:

* **14 of the original 25 apparent mixed meetings — 56% — were artefacts of source race-type errors**;
* **11 of 25 — 44% — survive as externally supported genuine mixed meetings**.

### Answer to the bounded database question

The database investigation began with:

> **Do the race-level `race_type_raw` values inside the 25 apparent mixed Flat/National Hunt Great Britain meetings appear correctly assigned?**

The answer is:

> **No.**

The source `race_type_raw` field contains material misclassifications within this anomaly population.

Fifteen race-level assignments were externally demonstrated to be wrong, including errors in both directions:

* races stored as `Flat` that were actually `Chase`, `Hurdle` or `NH Flat`;
* races stored as `Chase` or `Hurdle` that were actually Flat.

These errors are sufficient to create false meeting-level structural conclusions when `race_type_raw` is consumed without verification.

### Important methodological finding

The errors also show why race titles cannot safely be converted into race type using simple text rules.

Examples include races where words such as `Chase` or `Hurdles` appear in the title but do **not** describe the racing code, as well as genuine Chase, Hurdle and NH Flat races that the source nevertheless stores as `Flat`.

Title wording is therefore useful for **finding suspicious cases**, but not for automatically assigning corrected race type.

### Database consequence

The 15 externally verified corrections should be preserved as governed reconciliation evidence while leaving the original `race_type_raw` values immutable.

Until the next database release incorporates those corrections natively, analytical work that depends on race type should use the verified reconciliation layer rather than knowingly reverting to the contradicted source values.

The accepted Database v3 file itself remains unchanged.

### Consequence for the Great Britain structure study

The reader-facing study can now resume with a corrected interpretation.

It should not report that Great Britain contained 25 mixed Flat/National Hunt meetings.

The defensible result from this investigation is:

> **Within the original 25 apparent mixed meetings, 11 are externally supported genuine mixed programmes, all at Haydock; the other 14 were produced by source race-type classification errors.**

This resolves the database blocker that paused the study.


## 9. Widen the investigation: is `race_type_raw` reliable across Great Britain?

The mixed-meeting investigation has established that `race_type_raw` contains genuine classification errors.

However, that anomaly population was deliberately selected because it looked suspicious. It cannot tell us how reliable the field is across Great Britain as a whole.

Before searching for further contradictions or estimating an error rate, we must establish the complete Great Britain race-type population represented in Database v3.

### Question

> What values does `race_type_raw` contain across all Great Britain races, and how large is each population?

This step is descriptive only. It does not attempt to infer correct race type or remove known errors.


In [21]:
# Return to the complete accepted Database v3 Great Britain race population.
# We deliberately use the preserved source race_type_raw here because the
# purpose of this investigation is to assess that source field's reliability.
gb_race_type_population_sql = f"""
SELECT
    source_race_occurrence_id,
    raw_date,
    candidate_course_label,
    race_type_raw,
    race_name_raw
FROM {RACE_VIEW}
WHERE candidate_jurisdiction = 'Great Britain'
"""

with connect_read_only(DATABASE) as connection:
    gb_race_type_population = pd.read_sql_query(
        gb_race_type_population_sql,
        connection,
    )

# Protect the analytical grain: this investigation is race-level and should
# contain exactly one row per reconciled Great Britain race occurrence.
assert gb_race_type_population[
    "source_race_occurrence_id"
].is_unique, "Expected one row per Great Britain race occurrence"

# Preserve blanks explicitly rather than silently dropping them from the
# distribution. Missing or unexpected source labels are themselves evidence
# about the reliability and completeness of the field.
race_type_population_summary = (
    gb_race_type_population
    .assign(
        race_type_display=lambda df: (
            df["race_type_raw"].fillna("<NULL>")
        )
    )
    .groupby("race_type_display", as_index=False)
    .agg(
        races=("source_race_occurrence_id", "size"),
        earliest_date=("raw_date", "min"),
        latest_date=("raw_date", "max"),
    )
    .sort_values(
        ["races", "race_type_display"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

race_type_population_summary["share_pct"] = (
    race_type_population_summary["races"]
    / len(gb_race_type_population)
    * 100
)

print(
    "Great Britain races in race-type reliability population:",
    f"{len(gb_race_type_population):,}",
)
print(
    "Distinct race_type_raw values including NULL:",
    len(race_type_population_summary),
)

display(race_type_population_summary)

Great Britain races in race-type reliability population: 111,634
Distinct race_type_raw values including NULL: 4


,race_type_display,races,earliest_date,latest_date,share_pct
0,Flat,70218,2015-01-01,2026-05-27,62.900192
1,Hurdle,22645,2015-01-01,2026-05-27,20.285039
2,Chase,15671,2015-01-01,2026-05-27,14.037838
3,NH Flat,3100,2015-01-01,2026-05-26,2.776932


### What we found

Across all **111,634 Great Britain races** in Database v3, `race_type_raw` contains exactly four values:

* `Flat` — **70,218 races (62.90%)**;
* `Hurdle` — **22,645 races (20.29%)**;
* `Chase` — **15,671 races (14.04%)**;
* `NH Flat` — **3,100 races (2.78%)**.

There are:

* no `NULL` race types;
* no blank values;
* no additional or unexpected source categories.

The field is therefore **structurally complete** for the Great Britain population.

That does not establish that the assignments are correct. The previous investigation has already demonstrated genuine misclassification between these four categories.

The next step is therefore to search the entire Great Britain population for **internal title/type contradictions of the same kinds that exposed the confirmed errors**, while continuing to treat title wording only as a candidate-generation tool.

### Next question

> Across all 111,634 Great Britain races, how often does explicit race-title terminology contradict the stored `race_type_raw` value?


In [22]:
import re

# Work from the complete GB race-level population established above.
# Race-title wording is used only to identify candidates for investigation;
# it is not treated as authoritative race-type evidence.
gb_race_type_candidates = gb_race_type_population.copy()

# Use deliberately specific title markers that correspond to the three
# National Hunt source categories. These patterns are intended to find strong
# contradictions such as "Handicap Hurdle", "Hunters Chase", "NH Flat",
# "NHF Race" and "Bumper", while accepting that some lexical false positives
# will remain and must later be externally checked.
hurdle_pattern = re.compile(
    r"\b(?:hurdle|hurdles|hdl)\b",
    flags=re.IGNORECASE,
)

chase_pattern = re.compile(
    r"\b(?:chase|chases|chs)\b",
    flags=re.IGNORECASE,
)

nh_flat_pattern = re.compile(
    r"\b(?:nh\s*flat|nhf|national hunt flat|bumper)\b",
    flags=re.IGNORECASE,
)

gb_race_type_candidates["mentions_hurdle"] = (
    gb_race_type_candidates["race_name_raw"]
    .fillna("")
    .str.contains(hurdle_pattern)
)

gb_race_type_candidates["mentions_chase"] = (
    gb_race_type_candidates["race_name_raw"]
    .fillna("")
    .str.contains(chase_pattern)
)

gb_race_type_candidates["mentions_nh_flat"] = (
    gb_race_type_candidates["race_name_raw"]
    .fillna("")
    .str.contains(nh_flat_pattern)
)

# Flag only explicit title/type contradictions. This is intentionally
# asymmetric: a Flat-labelled race with an NH marker is suspicious, as is an
# NH-labelled race whose title explicitly describes a different NH category.
# Absence of a marker is not evidence that the stored type is wrong.
gb_race_type_candidates["lexical_contradiction"] = (
    (
        (gb_race_type_candidates["race_type_raw"] == "Flat")
        & (
            gb_race_type_candidates[
                ["mentions_hurdle", "mentions_chase", "mentions_nh_flat"]
            ].any(axis=1)
        )
    )
    |
    (
        (gb_race_type_candidates["race_type_raw"] == "Hurdle")
        & (
            gb_race_type_candidates["mentions_chase"]
            | gb_race_type_candidates["mentions_nh_flat"]
        )
    )
    |
    (
        (gb_race_type_candidates["race_type_raw"] == "Chase")
        & (
            gb_race_type_candidates["mentions_hurdle"]
            | gb_race_type_candidates["mentions_nh_flat"]
        )
    )
    |
    (
        (gb_race_type_candidates["race_type_raw"] == "NH Flat")
        & (
            gb_race_type_candidates["mentions_hurdle"]
            | gb_race_type_candidates["mentions_chase"]
        )
    )
)

lexical_contradiction_summary = (
    gb_race_type_candidates
    .groupby("race_type_raw", as_index=False)
    .agg(
        races=("source_race_occurrence_id", "size"),
        contradiction_candidates=("lexical_contradiction", "sum"),
    )
)

lexical_contradiction_summary["candidate_rate_pct"] = (
    lexical_contradiction_summary["contradiction_candidates"]
    / lexical_contradiction_summary["races"]
    * 100
)

print(
    "GB lexical contradiction candidates:",
    int(gb_race_type_candidates["lexical_contradiction"].sum()),
)

display(lexical_contradiction_summary)

# Also show the strongest candidate population so we can inspect its scale and
# composition before deciding whether to enumerate, stratify or externally
# sample it. Do not infer corrections from this table alone.
display(
    gb_race_type_candidates.loc[
        gb_race_type_candidates["lexical_contradiction"],
        [
            "raw_date",
            "candidate_course_label",
            "race_type_raw",
            "race_name_raw",
            "mentions_hurdle",
            "mentions_chase",
            "mentions_nh_flat",
        ],
    ]
    .sort_values(
        ["race_type_raw", "raw_date", "candidate_course_label"]
    )
    .reset_index(drop=True)
)

GB lexical contradiction candidates: 40


,race_type_raw,races,contradiction_candidates,candidate_rate_pct
0,Chase,15671,11,0.070193
1,Flat,70218,28,0.039876
2,Hurdle,22645,0,0.000000
3,NH Flat,3100,1,0.032258


,raw_date,candidate_course_label,race_type_raw,race_name_raw,mentions_hurdle,mentions_chase,mentions_nh_flat
0,2015-01-27,Taunton,Chase,Bulldog Chase Novices Hurdle (Div I),True,True,False
1,2015-01-27,Taunton,Chase,Bulldog Chase Novices Hurdle (Div II),True,True,False
2,2016-04-19,Kempton,Chase,Building Innovation Tapered Chase Mares Novice...,True,True,False
3,2019-01-09,Taunton,Chase,Eyes Down For The Cranmers Chase Handicap Hurdle,True,True,False
4,2021-10-05,Huntingdon,Chase,Peterborough Chase Day Sunday 5th December Nov...,True,True,False
5,2022-03-14,Taunton,Chase,William Hill Champion Chase Top Price Guarante...,True,True,False
6,2022-03-14,Taunton,Chase,William Hill Stayers Hurdle Top Price Guarante...,True,True,False
7,2023-03-13,Taunton,Chase,Williamhill Champion Chase Top Price Guarantee...,True,True,False
8,2023-03-13,Taunton,Chase,Williamhill Stayers Hurdle Top Price Guarantee...,True,True,False
9,2024-04-22,Hexham,Chase,Tay Valley Chasers Patrick Crofts Memorial Jun...,True,False,False


### What we found

The broad title/type contradiction search identified **40 candidate races** across the full population of 111,634 Great Britain races.

The candidates are rare relative to the whole field:

* `Flat`: 28 of 70,218;
* `Chase`: 11 of 15,671;
* `NH Flat`: 1 of 3,100;
* `Hurdle`: 0 of 22,645.

However, the candidate count is **not an error count**.

The detailed rows immediately show why. Many contain words such as `Chase` or `Hurdle` because of sponsors, promotional phrases or names rather than because those words describe the race type. Examples include `Chase Medical`, `Peterborough Chase Day`, `Champion Chase Top Price Guarantee` and `Chase Distillery`.

At the same time, this diagnostic successfully rediscovers many of the genuine errors already established during the mixed-meeting investigation.

### Important limitation

The previous investigation identified **15 confirmed wrong `race_type_raw` assignments**.

This lexical contradiction search appears to recover **12 of those 15**.

The three known errors it does not detect are particularly important:

* Bath — `Wild Berry Chase Handicap`, stored as `Chase` but actually Flat;
* Chepstow — `Hullabaloos Chase Handicap`, stored as `Chase` but actually Flat;
* Epsom — `No Hurdles ... Jump Jockeys Derby Handicap`, stored as `Hurdle` but actually Flat.

In each case, the misleading race title actually supports the incorrect stored type.

Therefore a title/type contradiction search can find genuine problems, but it **cannot measure the overall reliability of `race_type_raw`**. Some errors are invisible to this method.

The next step is to quantify exactly how the 40 candidates overlap with evidence we have already verified, then isolate the candidates that still require checking.


In [23]:
# Combine the two sets of externally verified corrections already established
# in this notebook. This gives us the complete set of 15 known wrong source
# assignments against which to measure the lexical diagnostic.
all_verified_race_type_corrections = pd.concat(
    [
        verified_race_type_corrections,
        additional_verified_race_type_corrections,
    ],
    ignore_index=True,
)

assert len(all_verified_race_type_corrections) == 15


# Restrict to the 40 candidates found by the full-GB lexical contradiction
# search. The raw source fields remain unchanged; we are only annotating the
# diagnostic result with evidence already established elsewhere in the notebook.
lexical_candidates = (
    gb_race_type_candidates.loc[
        gb_race_type_candidates["lexical_contradiction"]
    ]
    .copy()
    .reset_index(drop=True)
)


# Mark any candidate that is one of the 15 externally confirmed source errors.
# Date + governed course label + exact source title is sufficient here because
# those exact rows have already been resolved uniquely during the correction
# assertions above.
known_error_keys = set(
    all_verified_race_type_corrections[
        ["raw_date", "candidate_course_label", "race_name_raw"]
    ].itertuples(index=False, name=None)
)

lexical_candidates["verification_status"] = lexical_candidates.apply(
    lambda row: (
        "confirmed_error"
        if (
            row["raw_date"],
            row["candidate_course_label"],
            row["race_name_raw"],
        ) in known_error_keys
        else "unverified"
    ),
    axis=1,
)


# Cruises Chase Handicap was previously externally checked because its title
# looked contradictory, but it proved to be a genuine Flat race. Preserve that
# result explicitly as a known lexical false positive rather than asking for the
# same external verification again.
confirmed_correct_key = (
    "2024-07-12",
    "Chepstow",
    "Cruises Chase Handicap",
)

confirmed_correct_mask = (
    (lexical_candidates["raw_date"] == confirmed_correct_key[0])
    & (
        lexical_candidates["candidate_course_label"]
        == confirmed_correct_key[1]
    )
    & (
        lexical_candidates["race_name_raw"]
        == confirmed_correct_key[2]
    )
)

assert int(confirmed_correct_mask.sum()) == 1

lexical_candidates.loc[
    confirmed_correct_mask,
    "verification_status",
] = "confirmed_correct"


# Quantify both sides of the diagnostic:
# 1. how many known errors it successfully rediscovers;
# 2. how much of the 40-candidate set still needs external checking.
known_errors_detected = int(
    (lexical_candidates["verification_status"] == "confirmed_error").sum()
)

known_errors_missed = (
    len(all_verified_race_type_corrections)
    - known_errors_detected
)

candidate_status_summary = (
    lexical_candidates
    .groupby("verification_status", as_index=False)
    .agg(races=("source_race_occurrence_id", "size"))
    .sort_values("verification_status")
    .reset_index(drop=True)
)

print(
    "Previously confirmed errors:",
    len(all_verified_race_type_corrections),
)
print(
    "Confirmed errors detected by lexical diagnostic:",
    known_errors_detected,
)
print(
    "Confirmed errors missed by lexical diagnostic:",
    known_errors_missed,
)
print(
    "Lexical candidates still requiring verification:",
    int(
        (lexical_candidates["verification_status"] == "unverified").sum()
    ),
)

display(candidate_status_summary)

# Show only the unresolved candidates. These are the races that need external
# checking before this particular contradiction search can be closed.
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None,
    "display.width", 240,
):
    display(
        lexical_candidates.loc[
            lexical_candidates["verification_status"] == "unverified",
            [
                "raw_date",
                "candidate_course_label",
                "race_type_raw",
                "race_name_raw",
                "mentions_hurdle",
                "mentions_chase",
                "mentions_nh_flat",
            ],
        ].reset_index(drop=True)
    )

Previously confirmed errors: 15
Confirmed errors detected by lexical diagnostic: 12
Confirmed errors missed by lexical diagnostic: 3
Lexical candidates still requiring verification: 27


,verification_status,races
0,confirmed_correct,1
1,confirmed_error,12
2,unverified,27


,raw_date,candidate_course_label,race_type_raw,race_name_raw,mentions_hurdle,mentions_chase,mentions_nh_flat
0,2016-03-17,Chelmsford (AW),Flat,totetrifecta Pick The World Hurdle 1 2 3 Handicap,True,False,False
1,2016-06-15,Chelmsford (AW),Flat,Jem Partners Chase Handicap,False,True,False
2,2022-12-21,Lingfield (AW),Flat,Download The At The Races App Mares Open Maiden NHFlat Race (Category 3 Elimination) (GBB Race),False,False,True
3,2015-01-27,Taunton,Chase,Bulldog Chase Novices Hurdle (Div I),True,True,False
4,2015-01-27,Taunton,Chase,Bulldog Chase Novices Hurdle (Div II),True,True,False
5,2015-05-22,Bath,Flat,Simonstone Fiat 500X Chase Handicap,False,True,False
6,2015-08-15,Ripon,Flat,CHS Vehicles Maiden Auction Stakes,False,True,False
7,2015-08-15,Doncaster,Flat,Chase Medical Apprentice Handicap,False,True,False
8,2016-04-19,Kempton,Chase,Building Innovation Tapered Chase Mares Novices Hurdle,True,True,False
9,2016-08-13,Ripon,Flat,CHS Vehicles Maiden Auction Stakes,False,True,False


### External verification of the remaining 27 lexical candidates

All **27 previously unverified title/type contradiction candidates** were checked against published race results.

The checks identified:

* **10 additional incorrect `race_type_raw` assignments**;
* **17 correctly classified races whose titles merely triggered the lexical diagnostic**.

The 10 newly confirmed errors are:

| Date        | Course                                                            | Stored type | Verified type |
| ----------- | ----------------------------------------------------------------- | ----------: | ------------: |
| 27 Jan 2015 | Taunton — Bulldog Chase Novices Hurdle Div I                      |     `Chase` |    **Hurdle** |
| 27 Jan 2015 | Taunton — Bulldog Chase Novices Hurdle Div II                     |     `Chase` |    **Hurdle** |
| 19 Apr 2016 | Kempton — Building Innovation Tapered Chase Mares Novices Hurdle  |     `Chase` |    **Hurdle** |
| 9 Jan 2019  | Taunton — Eyes Down For The Cranmers Chase Handicap Hurdle        |     `Chase` |    **Hurdle** |
| 5 Oct 2021  | Huntingdon — Peterborough Chase Day... Novices Hurdle             |     `Chase` |    **Hurdle** |
| 14 Mar 2022 | Taunton — William Hill Champion Chase... Maiden Hurdle            |     `Chase` |    **Hurdle** |
| 21 Dec 2022 | Lingfield — Mares Open Maiden NHFlat Race                         |      `Flat` |   **NH Flat** |
| 13 Mar 2023 | Taunton — Williamhill Champion Chase... Maiden Hurdle             |     `Chase` |    **Hurdle** |
| 22 Apr 2024 | Hexham — Tay Valley Chasers... Junior National Hunt Hurdle        |     `Chase` |    **Hurdle** |
| 11 Nov 2025 | Huntingdon — Peterborough Chase... Junior National Hunt Flat Race |     `Chase` |   **NH Flat** |

Published results explicitly identify the two 2015 Taunton races and the 2016 Kempton race as hurdles. The 2019 Taunton and 2021 Huntingdon races are likewise documented as hurdle races despite being stored as `Chase`.

The same error pattern recurs at Taunton in 2022 and 2023: races whose sponsorship wording contains `Champion Chase` are actually maiden hurdles, while the neighbouring `Stayers Hurdle ... Handicap Chase` races really are chases.

The Lingfield race is externally identified as an NH Flat race, the Hexham race as a hurdle, and the 2025 Huntingdon race as a National Hunt Flat race.

### Confirmed lexical false positives

The other **17 candidates are correctly classified**.

They include several recurring reasons why simple keyword matching fails:

* sponsor or company names such as `Chase Medical`, `CHS Vehicles` and `Chase Distillery`;
* promotional references to other races such as `World Hurdle`, `Peterborough Chase`, `Champion Chase` and `Stayers Hurdle`;
* ordinary names containing `Chase`, such as `Jem Partners Chase`, `Simonstone Fiat 500X Chase`, `Castle Rock Harvest Pale Chase`, `Chase The Dream` and `West Chase`;
* `Chase GB` appearing as part of the name of a genuine National Hunt Flat race.

For example, the Chelmsford `World Hurdle` candidate was a Flat sprint on Polytrack, `Jem Partners Chase Handicap` was a Flat 1m2f handicap, and the Bath `Simonstone Fiat 500X Chase Handicap` was also a Flat race.

Similarly, the Ripon `CHS Vehicles` races and Doncaster `Chase Medical` races are ordinary Flat races, while the Hereford `Chase GB` race really was an NH Flat race.

### Result of the full lexical diagnostic

The entire 40-candidate population can now be classified:

* **22 confirmed race-type errors — 55%**;
* **18 correctly classified lexical false positives — 45%**.

This is an important result in both directions.

The diagnostic is useful: it found a substantial number of genuine source errors.

But it is **not a race-type parser**. Nearly half of its candidates are correctly classified races whose names contain misleading terminology.

It also fails to detect some known errors. Three of the original 15 confirmed errors — Bath, Chepstow and Epsom — were invisible to the contradiction rule because their titles actually reinforced the incorrect stored classification.

### Current state of evidence

Across the wider Great Britain investigation we have now externally established **25 incorrect `race_type_raw` assignments**:

* the original **15** errors found through the mixed-meeting investigation;
* **10 additional** errors found by the full-population lexical contradiction search.

Against the complete 111,634-race Great Britain population, those 25 known errors represent a **minimum observed error prevalence of about 0.022%**.

That figure is only a **lower bound**, not an estimated error rate.

The races were found through targeted anomaly detection rather than an unbiased sample, and we already know that some genuine errors are invisible to the lexical diagnostic.

### What this establishes

We can now say:

> `race_type_raw` is complete but not perfectly correct for Great Britain.

We cannot yet say:

> `race_type_raw` has an error rate of 0.022%.

Nor can we yet say that the field is sufficiently reliable for unrestricted analytical use.

To answer that larger question, we now need evidence that is **not selected because a race already looks suspicious**.

### Next question

> When Great Britain races are selected independently of title/type anomalies, how often is `race_type_raw` externally confirmed as correct?


In [24]:
# Preserve the ten additional errors established by the full-GB lexical
# candidate verification. These extend the 15 corrections already established
# by the mixed-meeting investigation; they do not overwrite race_type_raw.
additional_lexical_verified_corrections = pd.DataFrame(
    [
        ("2015-01-27", "Taunton", "Bulldog Chase Novices Hurdle (Div I)", "Chase", "Hurdle"),
        ("2015-01-27", "Taunton", "Bulldog Chase Novices Hurdle (Div II)", "Chase", "Hurdle"),
        ("2016-04-19", "Kempton", "Building Innovation Tapered Chase Mares Novices Hurdle", "Chase", "Hurdle"),
        ("2019-01-09", "Taunton", "Eyes Down For The Cranmers Chase Handicap Hurdle", "Chase", "Hurdle"),
        (
            "2021-10-05",
            "Huntingdon",
            "Peterborough Chase Day Sunday 5th December Novices Hurdle (GBB Race)",
            "Chase",
            "Hurdle",
        ),
        (
            "2022-03-14",
            "Taunton",
            "William Hill Champion Chase Top Price Guarantee Maiden Hurdle (GBB Race)",
            "Chase",
            "Hurdle",
        ),
        (
            "2022-12-21",
            "Lingfield (AW)",
            "Download The At The Races App Mares Open Maiden NHFlat Race (Category 3 Elimination) (GBB Race)",
            "Flat",
            "NH Flat",
        ),
        (
            "2023-03-13",
            "Taunton",
            "Williamhill Champion Chase Top Price Guarantee Maiden Hurdle (GBB Race)",
            "Chase",
            "Hurdle",
        ),
        (
            "2024-04-22",
            "Hexham",
            "Tay Valley Chasers Patrick Crofts Memorial Junior National Hunt Hurdle (GBB Race)",
            "Chase",
            "Hurdle",
        ),
        (
            "2025-11-11",
            "Huntingdon",
            "Book Now For The Peterborough Chase Fillies Junior National Hunt Flat Race (Cat 1 Elim) (GBB)",
            "Chase",
            "NH Flat",
        ),
    ],
    columns=[
        "raw_date",
        "candidate_course_label",
        "race_name_raw",
        "expected_raw_type",
        "verified_race_type",
    ],
)

assert len(additional_lexical_verified_corrections) == 10

# Combine every externally demonstrated race-type error found so far.
all_known_race_type_errors = pd.concat(
    [
        all_verified_race_type_corrections,
        additional_lexical_verified_corrections,
    ],
    ignore_index=True,
)

assert len(all_known_race_type_errors) == 25

print("Externally confirmed GB race-type errors so far:", len(all_known_race_type_errors))
display(
    all_known_race_type_errors.groupby(
        ["expected_raw_type", "verified_race_type"],
        as_index=False,
    ).size()
)

Externally confirmed GB race-type errors so far: 25


,expected_raw_type,verified_race_type,size
0,Chase,Flat,2
1,Chase,Hurdle,8
2,Chase,NH Flat,1
3,Flat,Chase,4
4,Flat,Hurdle,1
5,Flat,NH Flat,8
6,Hurdle,Flat,1


### What the known error directions show

The 25 externally confirmed errors are distributed across the stored source labels as follows:

* **13** races stored as `Flat`;
* **11** races stored as `Chase`;
* **1** race stored as `Hurdle`;
* **0** races stored as `NH Flat`.

The correction directions are:

* `Flat → Chase`: 4;
* `Flat → Hurdle`: 1;
* `Flat → NH Flat`: 8;
* `Chase → Flat`: 2;
* `Chase → Hurdle`: 8;
* `Chase → NH Flat`: 1;
* `Hurdle → Flat`: 1.

This demonstrates that the problem is not confined to a single mistaken mapping.

However, these counts **cannot be interpreted as comparative error rates for the four source labels**.

The known errors were found through mixed-meeting anomalies and title/type contradiction searches. Those methods were particularly capable of finding errors involving `Flat` and `Chase`, so the observed distribution is selection-biased.

We therefore need a new source of evidence that does not depend on a race looking suspicious.

## 10. Independent pilot sample

We will begin with a reproducible **stratified random pilot sample of 200 Great Britain races: 50 from each stored race type**.

Equal sampling by source label deliberately gives the smaller `Chase` and `NH Flat` populations enough representation for inspection.

The sample is selected using only `race_type_raw` and a fixed random seed. Race titles, mixed-meeting status and previous anomaly flags play no role in selection.

Because the sampling fractions differ between the four source labels, any eventual overall reliability estimate must be weighted back to the actual Great Britain race-type population.

This is a **pilot**, not yet the final reliability estimate. Its purpose is to determine whether errors also appear when races are selected independently of known anomalies and to inform how large the final validation sample needs to be.

### Question

> In an independently selected sample of Great Britain races, does external verification reveal further `race_type_raw` errors?


In [25]:
# Re-read the complete GB race population with governed course-local advertised
# time so each sampled race has enough context for subsequent external
# verification. Selection itself remains independent of title wording.
gb_random_sample_sql = f"""
SELECT
    source_race_occurrence_id,
    raw_date,
    candidate_course_label,
    advertised_start_course_local,
    temporal_resolution_status,
    race_type_raw,
    race_name_raw
FROM {RACE_VIEW}
WHERE candidate_jurisdiction = 'Great Britain'
"""

with connect_read_only(DATABASE) as connection:
    gb_random_sample_population = pd.read_sql_query(
        gb_random_sample_sql,
        connection,
    )

assert len(gb_random_sample_population) == 111_634
assert gb_random_sample_population[
    "source_race_occurrence_id"
].is_unique


# Sort before sampling so the fixed seed gives a reproducible selection even
# if the database query planner returns rows in a different physical order.
gb_random_sample_population = (
    gb_random_sample_population
    .sort_values("source_race_occurrence_id")
    .reset_index(drop=True)
)

# Select exactly 50 races independently from each stored source type.
# Do not exclude known errors, lexical candidates or mixed-meeting anomalies:
# doing so would make this a sample of "apparently clean" races rather than
# a sample of the race_type_raw field we are trying to assess.
PILOT_PER_TYPE = 50
PILOT_RANDOM_SEED = 20260810

gb_race_type_pilot = (
    gb_random_sample_population
    .groupby("race_type_raw", group_keys=False)
    .sample(
        n=PILOT_PER_TYPE,
        random_state=PILOT_RANDOM_SEED,
    )
    .copy()
)

assert len(gb_race_type_pilot) == 200

pilot_type_counts = (
    gb_race_type_pilot["race_type_raw"]
    .value_counts()
    .sort_index()
)

assert (pilot_type_counts == PILOT_PER_TYPE).all()


# Present a normal local racecard clock where the governed timestamp resolves.
# Do not substitute raw source `off` where temporal governance is unresolved.
gb_race_type_pilot["local_off_time"] = (
    gb_race_type_pilot[
        "advertised_start_course_local"
    ]
    .str.slice(11, 16)
    .fillna("UNRESOLVED")
)

# Give every sampled race a stable verification number so external checks can
# be recorded without relying on dataframe row positions.
gb_race_type_pilot = (
    gb_race_type_pilot
    .sort_values(
        [
            "race_type_raw",
            "raw_date",
            "candidate_course_label",
            "source_race_occurrence_id",
        ]
    )
    .reset_index(drop=True)
)

gb_race_type_pilot.insert(
    0,
    "verification_sample_id",
    [
        f"GBRT-{number:03d}"
        for number in range(1, len(gb_race_type_pilot) + 1)
    ],
)

print("Pilot sample races:", len(gb_race_type_pilot))
display(pilot_type_counts.rename("sampled_races"))

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None,
    "display.width", 260,
):
    display(
        gb_race_type_pilot[
            [
                "verification_sample_id",
                "raw_date",
                "candidate_course_label",
                "local_off_time",
                "race_type_raw",
                "race_name_raw",
            ]
        ]
    )

Pilot sample races: 200


race_type_raw
Chase      50
Flat       50
Hurdle     50
NH Flat    50
Name: sampled_races, dtype: int64

,verification_sample_id,raw_date,candidate_course_label,local_off_time,race_type_raw,race_name_raw
0,GBRT-001,2015-02-07,Newbury,14:25,Chase,Betfair Denman Chase
1,GBRT-002,2015-02-19,Sedgefield,16:45,Chase,Wills Property Services Handicap Chase
2,GBRT-003,2015-03-13,Cheltenham,15:20,Chase,Betfred Cheltenham Gold Cup Chase Grade 1
3,GBRT-004,2015-05-14,Fontwell,18:15,Chase,starsportsbet.co.uk Ladies Open Hunters Chase (for the Stuart Adamson Memorial Trophy)
4,GBRT-005,2015-10-05,Market Rasen,15:50,Chase,32Red.com Handicap Chase
5,GBRT-006,2015-10-22,Ludlow,16:45,Chase,Amateur Jockeys Association Amateur Riders Handicap Chase (for the Court of Hill Challenge Cup)
6,GBRT-007,2015-12-05,Chepstow,13:25,Chase,Angela Nettlefold Memorial In Aid of SSAFA Novices Limited Handicap Chase
7,GBRT-008,2016-04-02,Uttoxeter,16:55,Chase,Betfred Racings Biggest Supporter Handicap Chase
8,GBRT-009,2016-04-04,Wincanton,16:40,Chase,ApolloBet Fair Play Money Back Open Hunters Chase (for The John Dufosee Memorial Trophy)
9,GBRT-010,2016-12-13,Catterick,13:20,Chase,Come Racing New Years Day Beginners Chase


### What the pilot selection establishes

The independent pilot contains exactly **200 Great Britain races**, with **50 sampled from each stored `race_type_raw` category**:

* Chase — 50;
* Flat — 50;
* Hurdle — 50;
* NH Flat — 50.

The sample spans the full 2015–2026 period and was selected without reference to race titles, mixed-meeting anomalies or previous contradiction flags.

This is therefore suitable as an independent validation sample.

The race titles often appear consistent with the stored type, but **that observation must not be used as validation evidence**. The purpose of this sample is specifically to compare the stored value against an external source.

Before starting new external checks, we should identify whether random selection happened to include any races whose true type has already been externally established during the earlier investigation.

Any such overlap remains valid pilot evidence because the races were selected independently. We simply should not perform the same verification twice.

### Next question

> How many of the 200 independently selected races already have an externally verified race-type outcome from the investigation so far?


In [26]:
# Annotate the independently drawn pilot with the 25 source errors already
# established through external verification. Do NOT remove overlapping races:
# because selection occurred independently of the known-error process, any
# overlap is legitimate evidence within the random pilot.
known_error_lookup = (
    all_known_race_type_errors[
        [
            "raw_date",
            "candidate_course_label",
            "race_name_raw",
            "expected_raw_type",
            "verified_race_type",
        ]
    ]
    .drop_duplicates()
    .copy()
)

pilot_with_known_evidence = gb_race_type_pilot.merge(
    known_error_lookup,
    on=[
        "raw_date",
        "candidate_course_label",
        "race_name_raw",
    ],
    how="left",
    validate="one_to_one",
)

# A matched known error must still carry the same preserved raw source value
# that was externally contradicted earlier. Fail closed if identities match
# but the source classification does not.
known_overlap = pilot_with_known_evidence[
    "verified_race_type"
].notna()

assert (
    pilot_with_known_evidence.loc[
        known_overlap,
        "race_type_raw",
    ].to_numpy()
    ==
    pilot_with_known_evidence.loc[
        known_overlap,
        "expected_raw_type",
    ].to_numpy()
).all()

pilot_with_known_evidence["verification_status"] = "needs_external_check"

pilot_with_known_evidence.loc[
    known_overlap,
    "verification_status",
] = "previously_confirmed_error"

print("Pilot races:", len(pilot_with_known_evidence))
print(
    "Previously confirmed errors randomly selected into pilot:",
    int(known_overlap.sum()),
)
print(
    "Pilot races still requiring independent external verification:",
    int((~known_overlap).sum()),
)

if known_overlap.any():
    display(
        pilot_with_known_evidence.loc[
            known_overlap,
            [
                "verification_sample_id",
                "raw_date",
                "candidate_course_label",
                "race_type_raw",
                "verified_race_type",
                "race_name_raw",
            ],
        ]
    )

Pilot races: 200
Previously confirmed errors randomly selected into pilot: 0
Pilot races still requiring independent external verification: 200


### What we found

None of the **200 independently selected pilot races** overlaps with the 25 race-type errors already established through the anomaly investigations.

Therefore:

* previously confirmed errors randomly selected into the pilot: **0**;
* races still requiring external verification: **200**.

This is useful because the pilot is now entirely independent of the earlier error-discovery process.

Any error found in these 200 races will therefore be a **new error discovered through random sampling**, rather than a previously known anomaly appearing again by chance.

Conversely, races must not be treated as correct merely because their titles appear consistent with `race_type_raw`. Each pilot outcome must be based on an external result source.

### Verification method

For every sampled race we will record:

* the stored `race_type_raw`;
* the externally supported race type;
* whether the two agree;
* the external source used;
* a short evidence note sufficient to reconstruct the decision.

The verification will proceed in fixed `verification_sample_id` order so that the sample is not selectively checked according to how suspicious a race looks.

### Next question

> When these 200 independently selected races are externally verified, how often does the stored `race_type_raw` agree with the published race type?


In [27]:
# Create a verification ledger for the complete independently selected pilot.
# This keeps source values immutable while providing separate fields for the
# external judgement and its provenance.
pilot_verification = pilot_with_known_evidence[
    [
        "verification_sample_id",
        "raw_date",
        "candidate_course_label",
        "local_off_time",
        "race_type_raw",
        "race_name_raw",
    ]
].copy()

# Nothing in the random pilot has yet been scored. Use explicit pending states
# rather than inferring correctness from titles or other database fields.
pilot_verification["verified_race_type"] = pd.NA
pilot_verification["race_type_agrees"] = pd.NA
pilot_verification["verification_status"] = "pending"
pilot_verification["external_source"] = pd.NA
pilot_verification["external_reference"] = pd.NA
pilot_verification["evidence_note"] = pd.NA

assert len(pilot_verification) == 200
assert pilot_verification["verification_sample_id"].is_unique
assert (pilot_verification["verification_status"] == "pending").all()

print("Pilot verification records:", len(pilot_verification))
print(
    "Pending external checks:",
    int((pilot_verification["verification_status"] == "pending").sum()),
)

display(
    pilot_verification[
        [
            "verification_sample_id",
            "race_type_raw",
            "verification_status",
        ]
    ].head(10)
)

Pilot verification records: 200
Pending external checks: 200


,verification_sample_id,race_type_raw,verification_status
0,GBRT-001,Chase,pending
1,GBRT-002,Chase,pending
2,GBRT-003,Chase,pending
3,GBRT-004,Chase,pending
4,GBRT-005,Chase,pending
5,GBRT-006,Chase,pending
6,GBRT-007,Chase,pending
7,GBRT-008,Chase,pending
8,GBRT-009,Chase,pending
9,GBRT-010,Chase,pending


## 11. Independent pilot verification — GBRT-001 to GBRT-010

The first ten races were checked in fixed `verification_sample_id` order against published external result information.

All ten are stored as `Chase` in Database v3.

External result evidence independently identifies all ten as Chase races:

* **GBRT-001 — Newbury, 7 February 2015:** Chase;
* **GBRT-002 — Sedgefield, 19 February 2015:** Chase;
* **GBRT-003 — Cheltenham, 13 March 2015:** Chase;
* **GBRT-004 — Fontwell, 14 May 2015:** Chase;
* **GBRT-005 — Market Rasen, 5 October 2015:** Chase;
* **GBRT-006 — Ludlow, 22 October 2015:** Chase;
* **GBRT-007 — Chepstow, 5 December 2015:** Chase;
* **GBRT-008 — Uttoxeter, 2 April 2016:** Chase;
* **GBRT-009 — Wincanton, 4 April 2016:** Chase;
* **GBRT-010 — Catterick, 13 December 2016:** Chase.

### Result

> **10 checked; 10 agree with `race_type_raw`; 0 errors found.**

This is the first genuinely independent evidence from the random pilot.

It does not yet establish the reliability of the field. The sample must continue in its predetermined order, including the remaining Chase races and all three other stored race-type strata.


In [29]:
# Record the first fixed-order external-verification batch. The external
# references are result pages that independently identify the race type;
# race-title wording inside Database v3 is not being used as verification.
pilot_batch_001_010 = pd.DataFrame(
    [
        (
            "GBRT-001",
            "Chase",
            "Sky Sports",
            "https://www.skysports.com/racing/results/full-result/658996/newbury/07-02-2015/betfair-denman-chase",
            "Published result identifies the Grade 2 event as the Betfair Denman Chase.",
        ),
        (
            "GBRT-002",
            "Chase",
            "Sky Sports",
            "https://www.skysports.com/racing/results/full-result/660287/sedgefield/19-02-2015/wills-property-services-handicap-chase",
            "Published result identifies the event as a Handicap Chase.",
        ),
        (
            "GBRT-003",
            "Chase",
            "Racing Post",
            "https://www.racingpost.com/results/11/cheltenham/2015-03-13/598093",
            "Published result identifies the Gold Cup as a Chase and records 22 fences.",
        ),
        (
            "GBRT-004",
            "Chase",
            "HorseRacing.net",
            "https://www.horseracing.net/results/fontwell/14-05-15",
            "Published result classifies the Ladies Open Hunters Chase as Chase, Turf.",
        ),
        (
            "GBRT-005",
            "Chase",
            "HorseRacing.net",
            "https://www.horseracing.net/results/market-rasen/05-10-15",
            "Published result classifies the 15:50 32Red.com Handicap Chase as Chase, Turf.",
        ),
        (
            "GBRT-006",
            "Chase",
            "HorseRacing.net",
            "https://www.horseracing.net/results/ludlow/22-10-15",
            "Published result classifies the 16:45 Amateur Riders Handicap Chase as Chase, Turf.",
        ),
        (
            "GBRT-007",
            "Chase",
            "HorseRacing.net",
            "https://www.horseracing.net/results/chepstow/05-12-15",
            "Published result classifies the 13:25 Novices Limited Handicap Chase as Chase, Turf.",
        ),
        (
            "GBRT-008",
            "Chase",
            "HorseRacing.net",
            "https://www.horseracing.net/results/uttoxeter/02-04-16",
            "Published result classifies the Betfred Handicap Chase as Chase, Turf.",
        ),
        (
            "GBRT-009",
            "Chase",
            "HorseRacing.net",
            "https://www.horseracing.net/results/wincanton/04-04-16",
            "Published result classifies the 16:40 Open Hunters Chase as Chase, Turf.",
        ),
        (
            "GBRT-010",
            "Chase",
            "HorseRacing.net",
            "https://www.horseracing.net/results/catterick/13-12-16",
            "Published result classifies the 13:20 Beginners Chase as Chase, Turf.",
        ),
    ],
    columns=[
        "verification_sample_id",
        "verified_race_type",
        "external_source",
        "external_reference",
        "evidence_note",
    ],
)

assert len(pilot_batch_001_010) == 10
assert pilot_batch_001_010["verification_sample_id"].is_unique


# Update only rows belonging to this predetermined batch. Fail closed if any
# sampled row has already been scored or if its preserved source type differs
# from the externally established type unexpectedly.
for result in pilot_batch_001_010.itertuples(index=False):
    mask = (
        pilot_verification["verification_sample_id"]
        == result.verification_sample_id
    )

    assert int(mask.sum()) == 1, (
        f"Expected exactly one pilot row for {result.verification_sample_id}"
    )

    assert (
        pilot_verification.loc[mask, "verification_status"].iloc[0]
        == "pending"
    ), f"{result.verification_sample_id} has already been verified"

    raw_type = pilot_verification.loc[mask, "race_type_raw"].iloc[0]

    pilot_verification.loc[mask, "verified_race_type"] = (
        result.verified_race_type
    )
    pilot_verification.loc[mask, "race_type_agrees"] = (
        raw_type == result.verified_race_type
    )
    pilot_verification.loc[mask, "verification_status"] = "verified"
    pilot_verification.loc[mask, "external_source"] = result.external_source
    pilot_verification.loc[mask, "external_reference"] = (
        result.external_reference
    )
    pilot_verification.loc[mask, "evidence_note"] = result.evidence_note


# Summarise evidence actually collected so far. Do not extrapolate from this
# incomplete batch to the full population or even to the full Chase stratum.
verified_so_far = pilot_verification.loc[
    pilot_verification["verification_status"] == "verified"
].copy()

print("Pilot races externally verified:", len(verified_so_far))
print(
    "Agreements:",
    int(verified_so_far["race_type_agrees"].eq(True).sum()),
)
print(
    "Disagreements:",
    int(verified_so_far["race_type_agrees"].eq(False).sum()),
)
print(
    "Still pending:",
    int(
        pilot_verification["verification_status"]
        .eq("pending")
        .sum()
    ),
)

display(
    verified_so_far[
        [
            "verification_sample_id",
            "race_type_raw",
            "verified_race_type",
            "race_type_agrees",
            "external_source",
        ]
    ]
)

Pilot races externally verified: 10
Agreements: 10
Disagreements: 0
Still pending: 190


,verification_sample_id,race_type_raw,verified_race_type,race_type_agrees,external_source
0,GBRT-001,Chase,Chase,True,Sky Sports
1,GBRT-002,Chase,Chase,True,Sky Sports
2,GBRT-003,Chase,Chase,True,Racing Post
3,GBRT-004,Chase,Chase,True,HorseRacing.net
4,GBRT-005,Chase,Chase,True,HorseRacing.net
5,GBRT-006,Chase,Chase,True,HorseRacing.net
6,GBRT-007,Chase,Chase,True,HorseRacing.net
7,GBRT-008,Chase,Chase,True,HorseRacing.net
8,GBRT-009,Chase,Chase,True,HorseRacing.net
9,GBRT-010,Chase,Chase,True,HorseRacing.net


### Independent pilot result

The independent reliability pilot selected **200 Great Britain races** without reference to race titles, mixed-meeting anomalies or previously identified classification problems.

The sample contained exactly 50 races from each stored `race_type_raw` category:

* `Chase`;
* `Flat`;
* `Hurdle`;
* `NH Flat`.

None of the sampled races overlapped with the 25 race-type errors already identified through the targeted investigations.

All 200 sampled races were then checked against external published result evidence.

### Result

External verification found:

* **200 agreements with `race_type_raw`;**
* **0 disagreements;**
* **0 newly discovered race-type errors.**

Every source-type stratum returned 50 agreements from 50 independently selected races.

### Interpretation

This materially changes what we can say about the reliability of the field.

The targeted investigations proved that `race_type_raw` is **not perfectly correct**. We have already externally demonstrated 25 genuine source classification errors, including errors between Flat, Hurdle, Chase and NH Flat.

However, the independent random pilot found **no additional errors in 200 races selected without regard to whether they looked suspicious**.

The combined evidence therefore supports a distinction between two claims:

> `race_type_raw` is not error-free.

and:

> The errors do not appear to be pervasive across ordinary Great Britain race records.

The field consequently appears **highly reliable for broad descriptive analysis of Great Britain racing**, provided that known externally verified corrections are reconciled.

It should not, however, be treated as unquestionable in analyses where a single incorrect race type can materially change the conclusion. The mixed-meeting investigation demonstrated exactly that failure mode: a small number of individual errors created 14 entirely false mixed-code meetings.

### Statistical limitation

The observed disagreement rate in this pilot is **0%**, but that is not evidence that the true population error rate is zero.

With no errors observed in 200 races, a simple common-rate binomial calculation gives a one-sided 95% upper bound of approximately **1.49%**. Because the pilot deliberately sampled equal numbers from each stored race type rather than sampling Great Britain races in population proportion, that figure should be treated only as a sensitivity calculation rather than as a final design-adjusted population error estimate.

The important evidential result is therefore not “the error rate is zero”.

It is:

> **A targeted search can find real `race_type_raw` errors, but an independent 200-race pilot found no further errors. The field appears generally reliable rather than systematically unreliable.**

### Database conclusion

For Great Britain, `race_type_raw` is suitable for analytical use **with reconciliation of the known errors and explicit caution in anomaly-sensitive applications**.

The original raw source values should remain immutable for lineage.

Externally demonstrated corrections should be exposed separately through the governed reconciliation layer, and analyses capable of being materially affected by an isolated classification error should retain an anomaly-review step.

A substantially larger independent validation sample would be required only if the project later needs a precise estimate of a very low residual race-type error rate.


In [34]:
from pathlib import Path
import pandas as pd

# Load the completed external-verification ledger for the independent
# 200-race Great Britain race-type pilot. This is the audited version in
# which every sampled race has its own external evidence locator.
PILOT_VERIFICATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "gb_race_type_pilot_verification_200_verified.csv"
)

pilot_verification_completed = pd.read_csv(
    PILOT_VERIFICATION_PATH,
    dtype={
        "verification_sample_id": "string",
        "raw_date": "string",
        "candidate_course_label": "string",
        "local_off_time": "string",
        "race_type_raw": "string",
        "race_name_raw": "string",
        "verified_race_type": "string",
        "verification_status": "string",
        "primary_evidence_url": "string",
        "secondary_audit_url": "string",
        "evidence_note": "string",
    },
)

# Protect the exact pilot population established earlier in the notebook.
# The persisted ledger must contain one and only one result for every sampled
# race, with no additions or omissions.
assert len(pilot_verification_completed) == 200

assert pilot_verification_completed[
    "verification_sample_id"
].is_unique

assert set(
    pilot_verification_completed["verification_sample_id"]
) == set(
    gb_race_type_pilot["verification_sample_id"]
)

# Every row must represent a completed external verification. Under the
# project's provenance rule, a row does not count as verified unless it also
# records a reconstructible external evidence locator and evidence note.
assert (
    pilot_verification_completed["verification_status"]
    == "verified_external"
).all()

assert pilot_verification_completed[
    "verified_race_type"
].notna().all()

assert (
    pilot_verification_completed["primary_evidence_url"]
    .str.startswith(("http://", "https://"), na=False)
    .all()
)

assert (
    pilot_verification_completed["evidence_note"]
    .str.strip()
    .ne("")
    .all()
)

# Recalculate agreement from the preserved database value and the separately
# recorded externally verified value. Do not trust a persisted agreement flag
# when the result can be reconstructed directly from the evidence fields.
pilot_verification_completed["race_type_agrees_recomputed"] = (
    pilot_verification_completed["race_type_raw"]
    == pilot_verification_completed["verified_race_type"]
)

pilot_result_by_type = (
    pilot_verification_completed
    .groupby("race_type_raw", as_index=False)
    .agg(
        sampled_races=("verification_sample_id", "size"),
        agreements=("race_type_agrees_recomputed", "sum"),
    )
)

pilot_result_by_type["disagreements"] = (
    pilot_result_by_type["sampled_races"]
    - pilot_result_by_type["agreements"]
)

print(
    "Externally verified pilot races:",
    len(pilot_verification_completed),
)
print(
    "Agreements:",
    int(
        pilot_verification_completed[
            "race_type_agrees_recomputed"
        ].sum()
    ),
)
print(
    "Disagreements:",
    int(
        (
            ~pilot_verification_completed[
                "race_type_agrees_recomputed"
            ]
        ).sum()
    ),
)

display(pilot_result_by_type)

Externally verified pilot races: 200
Agreements: 200
Disagreements: 0


,race_type_raw,sampled_races,agreements,disagreements
0,Chase,50,50,0
1,Flat,50,50,0
2,Hurdle,50,50,0
3,NH Flat,50,50,0


### What we found

The independent stratified pilot externally verified **200 Great Britain races**:

* 50 stored as `Chase`;
* 50 stored as `Flat`;
* 50 stored as `Hurdle`;
* 50 stored as `NH Flat`.

All **200 externally verified race types agreed with `race_type_raw`**.

No new race-type errors were found in the independent pilot.

### Interpretation

This provides evidence from a population selected independently of the anomaly searches.

The earlier targeted investigation proved that `race_type_raw` is not error-free: **25 genuine classification errors** have been externally established.

However, the independent pilot found:

> **200 agreements from 200 externally checked races, with zero disagreements across all four stored race-type categories.**

The combined evidence therefore supports the conclusion that:

> **`race_type_raw` is generally reliable for Great Britain analytical use, but contains rare, material edge-case errors.**

Those errors matter particularly in analyses where a single incorrect race classification can change the interpretation of a meeting, as demonstrated by the original mixed Flat/National Hunt investigation.

The pilot does not provide a precise estimate of the residual error rate and should not be used to claim that `race_type_raw` is perfectly accurate.

### Database decision

No wider replacement or reconstruction of `race_type_raw` is justified by the evidence.

The appropriate treatment is to:

* preserve the original source `race_type_raw` unchanged for lineage;
* retain the **25 externally verified corrections** as governed reconciliation evidence;
* expose those verified corrections to analytical studies;
* retain anomaly review for questions that are particularly sensitive to isolated race-type errors.

This is sufficient to close the wider Great Britain `race_type_raw` reliability investigation and return to the study that triggered it.
